# MiniFly mechanism sweep

Run all cells in a standard Colab CPU session. The notebook contains the code and compact FlyWire inputs. The default **smoke** profile executes all 33 arms at one seed. Set **PROFILE = "screen"** in the setup cell for the 99-run, three-seed development roster. Use **pilot** for the three-seed roster at a shorter dose.

The working-memory, credit, replay, and temporal-learning blocks use compact analogues. Cortical feedback reuses Claude’s mod-97 learner. The mushroom-body and central-complex probes use the bundled v630 extract. The grid probe uses engineered toroidal transport. Attention is a supervised reference. These are development comparisons. A smoke result measures execution and pairing.

The notebook uses the runtime's installed NumPy and PyTorch, installing a missing package if needed. Outputs are saved per job. Turn on the optional Drive setting before running if you want receipts to survive a Colab reset. Rerunning the same profile, settings and software environment reuses completed jobs. Interrupted jobs start again.


In [ ]:
# Change these settings, then Run all.
PROFILE = "smoke"  # smoke, pilot, or screen
WORKERS = 2
SAVE_TO_DRIVE = False
BLOCKS = []  # Empty runs all eleven. Example: ["cortical", "attention"]

from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import os
import subprocess
import sys

base = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORK = base / "_minifly_sweep_runtime"
WORK.mkdir(parents=True, exist_ok=True)
(WORK / "cx").mkdir(exist_ok=True)
OUT = base / "MiniFlyReceipts"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/MiniFlyReceipts")
OUT.mkdir(parents=True, exist_ok=True)

for package in ("numpy", "torch"):
    try:
        print(f"{package} {version(package)}")
    except PackageNotFoundError:
        index = ["--index-url", "https://download.pytorch.org/whl/cpu"] if package == "torch" else []
        subprocess.run([sys.executable, "-m", "pip", "install", *index, package], check=True)

os.chdir(WORK)
print(f"Python {sys.version.split()[0]}. Receipts in {OUT}.")


## Embedded source and data

The following cells materialize the notebook's own source and verify the FlyWire input hashes. The retained anatomy and prior calibration come from the [MiniFly v630 engineering bundle](https://github.com/ChangweiZhou/digitalbaby/tree/67b7b2f61cc8cec41ce952fa9b4eac68984d630b/output/minifly-connectome-navigation). The arithmetic learner comes from [Claude’s archived mod-97 implementation](https://github.com/ChangweiZhou/digitalbaby/tree/249e17acadeeab475ca006e956d18ed149f94ec1/MOD97_LOCAL_RULE_20261106).


In [ ]:
%%writefile mod97_local.py
"""Mod-97 addition from raw bytes: backprop versus V181-style direct-feedback local rule.

Task, split, rendering, model shape, loss and scoring follow Codex's
minifly_three_remaining_experiments_20261106 mod-97 run (fixtures.py, models.py,
official_mod97.py at commit 501ca0c). The KC association residual is omitted
(it contributed 0.9-1.6% of held sums in that run). The only difference between
arms is how the weight update is computed:

  bp       autograd backpropagation
  dfa      direct feedback alignment as in V181's learner.py: the output error is
           projected to each hidden layer by a fixed random matrix, and every weight
           update is local (presynaptic activity x projected error). The embedding
           receives its own fixed random projection of the output error.
  dfa_fix  same as dfa, embedding frozen at its random birth (fixed input code).

All arms use Adam lr 0.003, batch 64, no weight decay, same data order per seed.
"""
import os
for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[k] = "1"
import json
import math
import sys
import time
from itertools import combinations_with_replacement, islice
from pathlib import Path
from random import Random

import torch as T
from torch import nn
from torch.nn.functional import cross_entropy, pad

T.set_num_threads(1)
EOS = 256
BOS = 257
SPLIT_SEED = 57467          # Codex fixtures.SEED
SEEDS = (73129, 93683, 116089)
BATCH = 64
BLOCK = 512
OUT = Path(__file__).resolve().parent / "runs"


# ---------------------------------------------------------------- task (verbatim logic from fixtures.py)
def carry(pair):
    a, b = pair
    return int(a % 10 + b % 10 >= 10)


def mod_split(seed=SPLIT_SEED):
    rng = Random(seed)
    diag = set(rng.sample(range(97), 48))
    extra = set(rng.sample(range(97), 48))
    out = [[], [], []]
    for residue in range(97):
        groups = [[p for p in combinations_with_replacement(range(97), 2)
                   if sum(p) % 97 == residue and p[0] != p[1] and carry(p) == c] for c in range(2)]
        d = (residue * 49) % 97
        n = 24 + (residue in extra) - (residue in diag)
        first = round(n * len(groups[0]) / 48)
        for g, k in zip(groups, (first, n - first)):
            rng.shuffle(g)
            out[0].extend(g[:k])
            rest = g[k:]
            side = 1 if len(out[1]) <= len(out[2]) else 2
            out[side].extend(rest[::2])
            out[3 - side].extend(rest[1::2])
        side = 1 if len(out[1]) <= len(out[2]) else 2
        out[0 if residue in diag else side].append((d, d))
    return out


def expand(pairs):
    return [p for a, b in pairs for p in (((a, b), (b, a)) if a != b else ((a, b),))]


def arithmetic(pair, mod=97):
    a, b = pair
    prompt = f"{a:02d}+{b:02d}".encode() + b"="
    y = (a + b) % mod
    return list(prompt) + list(f"{y:02d}".encode()) + [EOS]


def panels(seed):
    return tuple(map(expand, mod_split(seed)))


def stream(train, seed):
    targets = [arithmetic(p) for p in train]
    order = Random(seed + 1)
    chunk = []
    while True:
        for row in order.sample(targets, len(targets)):
            chunk.append(row)
            if len(chunk) == BATCH:
                yield chunk
                chunk = []


# ---------------------------------------------------------------- model (shape from models.Modular, shared path only)
class Net(nn.Module):
    def __init__(self, W=8, H=128):
        super().__init__()
        self.W, self.H = W, H                        # byte window (8 in rounds 1-3 and Experiment 1); hidden width (128 before the width test)
        self.book = nn.Embedding(258, 8, padding_idx=BOS)
        self.l1 = nn.Linear(8 * W, H)
        self.l2 = nn.Linear(H, H)
        self.l3 = nn.Linear(H, 257)

    def window(self, x):
        return self.book(pad(x, (self.W - 1, 0), value=BOS).unfold(1, self.W, 1)).flatten(2)

    def forward(self, x, keep=False, drop=0., rng=None):
        u = self.window(x)
        a1 = self.l1(u); z1 = T.tanh(a1)
        a2 = self.l2(z1); z2 = T.tanh(a2)
        mask = None
        if drop > 0:   # inverted dropout on the second hidden layer, as V181's T layer
            mask = (T.rand(z2.shape, generator=rng) >= drop).float() / (1 - drop)
            z2 = z2 * mask
        logits = self.l3(z2)
        if keep:
            return logits, (u, z1, z2, mask)
        return logits


V181_OPT = dict(lr=.001, weight_decay=1e-4)   # V181 learner.py: lr .001, coupled L2 1e-4, dropout .1 on 2nd hidden
DROP = {"bp_v181": .1, "dfa_v181": .1}


def make(seed, arm="bp"):
    T.manual_seed(seed)
    m = Net()
    g = T.Generator().manual_seed(seed + 99)
    F = {  # fixed random feedback, scale as V181 (1/sqrt(257))
        "F2": T.randn(257, 128, generator=g) / math.sqrt(257),
        "F1": T.randn(257, 128, generator=g) / math.sqrt(257),
        "F0": T.randn(257, 64, generator=g) / math.sqrt(257),
    }
    F["drop"] = DROP.get(arm, 0.)
    F["rng"] = T.Generator().manual_seed(seed + 7)
    F["mode"] = "sign" if arm.startswith("sign") else "dfa"
    if arm.startswith("bpw") or arm.startswith("signw"):   # decoupled weight decay (AdamW), lr .001, as in the grokking papers
        wd = WD[arm[-2:]]
        opt = T.optim.AdamW(m.parameters(), lr=.001, weight_decay=wd, betas=(.9, .98))
    elif arm.endswith("_v181"):
        opt = T.optim.Adam(m.parameters(), **V181_OPT)
    elif arm == "dfa_outwd":      # weight decay on the output layer only
        out = [m.l3.weight, m.l3.bias]
        rest = [p for n, p in m.named_parameters() if not n.startswith("l3.")]
        opt = T.optim.Adam([dict(params=rest, weight_decay=0.), dict(params=out, weight_decay=1e-4)], lr=.003)
    elif arm == "dfa_lowlr":      # lower learning rate, no decay
        opt = T.optim.Adam(m.parameters(), lr=.001)
    else:
        opt = T.optim.Adam(m.parameters(), lr=.003)
    return m, F, opt


def rows_tensor(rows):
    return T.tensor([[BOS, *r] for r in rows])


def update_bp(m, F, opt, rows):
    data = rows_tensor(rows)
    logits = m(data[:, :-1], drop=F["drop"], rng=F["rng"])
    loss = cross_entropy(logits.flatten(0, 1), data[:, 1:].flatten())
    opt.zero_grad()
    loss.backward()
    opt.step()
    return float(loss.detach())


@T.no_grad()
def local_grads(m, F, rows, freeze_emb):
    """Direct feedback alignment. Returns dict of grads keyed by parameter name."""
    data = rows_tensor(rows)
    x = data[:, :-1]
    y = data[:, 1:].flatten()
    logits, (u, z1, z2, mask) = m(x, keep=True, drop=F["drop"], rng=F["rng"])
    N = y.numel()
    L = logits.flatten(0, 1)
    e = L.softmax(1)
    e[T.arange(N), y] -= 1
    e /= N                                   # d loss / d logits (mean CE)
    loss = float(-(L.log_softmax(1)[T.arange(N), y]).mean())
    u = u.flatten(0, 1); z1 = z1.flatten(0, 1); z2 = z2.flatten(0, 1)
    g = {}
    g["l3.weight"] = e.T @ z2
    g["l3.bias"] = e.sum(0)
    sign = F.get("mode") == "sign"

    def S(W):   # sign-symmetric feedback: sign of the forward weight, scaled to its mean magnitude
        return W.sign() * W.abs().mean()

    if mask is None:
        tp2 = 1 - z2 * z2
        d2 = (e @ (S(m.l3.weight) if sign else F["F2"])) * tp2
    else:
        mask = mask.flatten(0, 1)
        pre = z2 / mask.clamp_min(1e-6) * (mask > 0)   # undo the scaling where kept
        d2 = (e @ (S(m.l3.weight) if sign else F["F2"])) * (1 - pre * pre) * mask
    g["l2.weight"] = d2.T @ z1
    g["l2.bias"] = d2.sum(0)
    d1 = ((d2 @ S(m.l2.weight)) if sign else (e @ F["F1"])) * (1 - z1 * z1)
    g["l1.weight"] = d1.T @ u
    g["l1.bias"] = d1.sum(0)
    if not freeze_emb:
        d0 = ((d1 @ S(m.l1.weight)) if sign else (e @ F["F0"])).reshape(N, m.W, 8)  # per slot, per embedding dim
        tok = pad(x, (m.W - 1, 0), value=BOS).unfold(1, m.W, 1).flatten(0, 1)   # N x W token ids
        ge = T.zeros(258, 8)
        ge.index_add_(0, tok.flatten(), d0.reshape(-1, 8))
        ge[BOS] = 0
        g["book.weight"] = ge
    return g, loss


def update_dfa(m, F, opt, rows, freeze_emb=False):
    g, loss = local_grads(m, F, rows, freeze_emb)
    opt.zero_grad()
    for name, p in m.named_parameters():
        p.grad = g[name] if name in g else None   # frozen embedding: no grad, Adam skips it
    opt.step()
    return loss


# ---------------------------------------------------------------- scoring (as official_mod97.score)
@T.no_grad()
def respond(m, prompts, cap=5, batch=256):
    outs = []
    for i in range(0, len(prompts), batch):
        p = T.tensor([[BOS, *x] for x in prompts[i:i + batch]])
        done = T.zeros(len(p), dtype=T.bool)
        gen = [[] for _ in range(len(p))]
        for _ in range(cap):
            y = m(p)[:, -1].argmax(1)
            for j in range(len(p)):
                if not done[j]:
                    gen[j].append(int(y[j]))
                    if int(y[j]) == EOS:
                        done[j] = True
            p = T.cat([p, y[:, None]], 1)
            if bool(done.all()):
                break
        outs.extend(gen)
    return outs


def score(m, pairs):
    rows = [arithmetic(p) for p in pairs]
    prompts = [r[:6] for r in rows]
    gold = [r[6:] for r in rows]
    outs = respond(m, prompts)
    exact = {p: o == g for p, o, g in zip(pairs, outs, gold)}
    return dict(
        n=len(pairs),
        exact=sum(exact.values()),
        first=sum(o[:1] == g[:1] for o, g in zip(outs, gold)),
        eos=sum(EOS in o for o in outs),
        both_orders=sum(exact[a, b] and exact[b, a] for a, b in pairs if a < b),
        both_order_n=sum(a < b for a, b in pairs),
        wrap=[sum(exact[a, b] for a, b in pairs if a + b >= 97), sum(a + b >= 97 for a, b in pairs)],
        decimal_carry=[sum(exact[a, b] for a, b in pairs if a % 10 + b % 10 >= 10),
                       sum(a % 10 + b % 10 >= 10 for a, b in pairs)],
    )


# ---------------------------------------------------------------- qualification
def qualify():
    train, monitor, sealed = panels(SEEDS[0])
    assert len(train) == 4704 and len(monitor) + len(sealed) == 4705, (len(train), len(monitor), len(sealed))
    allp = set(train) | set(monitor) | set(sealed)
    assert len(allp) == 97 * 97 and not (set(train) & set(monitor)) and not (set(train) & set(sealed))
    assert all(int(bytes(arithmetic(p)[6:8])) == sum(p) % 97 for p in allp)
    # swapped form of a held pair is also held
    assert all((b, a) in sealed for a, b in sealed)
    # output-layer and layer-2 local grads reduce to autograd when F2 = W3
    m, F, opt = make(1)
    rows = list(islice(stream(train, 1), 1))[0]
    F["F2"] = m.l3.weight.detach().clone()
    g, _ = local_grads(m, F, rows, False)
    data = rows_tensor(rows)
    loss = cross_entropy(m(data[:, :-1]).flatten(0, 1), data[:, 1:].flatten())
    loss.backward()
    for name in ("l3.weight", "l3.bias", "l2.weight", "l2.bias"):
        p = dict(m.named_parameters())[name]
        assert T.allclose(p.grad, g[name], atol=1e-6, rtol=1e-4), name
    # true backprop d1 is NOT of the DFA form, so l1 is expected to differ; record the cosine
    cos = T.nn.functional.cosine_similarity(dict(m.named_parameters())["l1.weight"].grad.flatten(), g["l1.weight"].flatten(), 0)
    # short training smoke: 32 updates each arm, finite
    res = {}
    for arm in ("bp", "dfa", "dfa_fix"):
        m, F, opt = make(1)
        for rows in islice(stream(train, 1), 32):
            l = (update_bp if arm == "bp" else lambda *a: update_dfa(*a, freeze_emb=(arm == "dfa_fix")))(m, F, opt, rows)
            assert math.isfinite(l)
        res[arm] = dict(loss=l, taught=score(m, train[:256])["exact"])
    out = dict(split=[len(train), len(monitor), len(sealed)], l1_cosine_bp_vs_dfa_with_F2_eq_W3=float(cos), smoke=res)
    OUT.mkdir(exist_ok=True)
    (OUT / "QUALIFY.json").write_text(json.dumps(out, indent=1))
    print(json.dumps(out, indent=1))


# ---------------------------------------------------------------- main run
WD = {"01": .1, "03": .3, "10": 1.}
ARMS = ("bp", "dfa", "dfa_fix", "bp_v181", "dfa_v181", "dfa_outwd", "dfa_lowlr", "sign",
        "bpw01", "bpw03", "bpw10", "signw10")


def subsample(train, frac, seed):
    """Keep a fraction of ALL 4,753 unordered pairs as taught, drawn from the taught groups; both orders stay together."""
    groups = sorted({(min(a, b), max(a, b)) for a, b in train})
    k = round(frac * 4753)
    assert k <= len(groups), (k, len(groups))
    rng = Random(seed + 500 + round(frac * 1000))
    keep = rng.sample(groups, k)
    return expand(sorted(keep))


def run(arm, seed, dose, frac=.5):
    OUT.mkdir(exist_ok=True)
    path = OUT / (f"{arm}_{seed}.json" if frac == .5 else f"{arm}_{seed}_f{round(frac * 100)}.json")
    assert arm in ARMS, arm
    train, monitor, sealed = panels(seed)
    if frac != .5:
        train = subsample(train, frac, seed)
    m, F, opt = make(seed, arm)
    upd = update_bp if arm.startswith("bp") else (lambda *a: update_dfa(*a, freeze_emb=(arm == "dfa_fix")))
    batches = stream(train, seed)
    curve = []
    t0 = time.process_time()
    for stop in range(BLOCK, dose + 1, BLOCK):
        losses = [upd(m, F, opt, rows) for rows in islice(batches, BLOCK)]
        row = dict(step=stop, loss=sum(losses) / len(losses), taught=score(m, train), monitor=score(m, monitor),
                   sealed=score(m, sealed), cpu=time.process_time() - t0)
        curve.append(row)
        print(arm, seed, stop, f"loss {row['loss']:.3f}", f"taught {row['taught']['exact']}/{row['taught']['n']}",
              f"monitor {row['monitor']['exact']}/{row['monitor']['n']}", f"sealed {row['sealed']['exact']}/{row['sealed']['n']}",
              f"cpu {row['cpu']:.0f}s", flush=True)
        path.write_text(json.dumps(dict(arm=arm, seed=seed, dose=dose, frac=frac, taught_n=len(train),
                                        params=sum(p.numel() for p in m.parameters()), curve=curve), indent=1))


if __name__ == "__main__":
    if sys.argv[1] == "qualify":
        qualify()
    else:
        run(sys.argv[1], int(sys.argv[2]), int(sys.argv[3]), float(sys.argv[4]) if len(sys.argv) > 4 else .5)


In [ ]:
%%writefile cx/qualify.py
"""Fixed-gain and analytic fixture checks. No fitted model or closed-loop learner."""
from collections import defaultdict
from hashlib import sha256
import json
from pathlib import Path
import sys
import time
import numpy as np

ROOT=Path(__file__).resolve().parent
GOALS=np.array([[.30,.50],[.70,.50]])
SEED=61317


def trajectory(theta,moves):
    h=float(theta);xy=np.zeros(2)
    for yaw,advance in moves:
        h+=yaw
        xy+=advance*np.array([np.cos(h),np.sin(h)])
    return h,xy


def arena():
    rng=np.random.default_rng(SEED)
    starts=np.column_stack([rng.uniform(.43,.57,32),np.repeat([.16,.84],16)])
    heading=rng.uniform(-np.pi,np.pi,32)
    visits=np.column_stack([np.linspace(.24,.76,53),np.full(53,.5)])
    assert np.min(np.linalg.norm(starts[:,None]-visits[None],axis=2))>.25
    def run(s,h,g):
        xy=s.copy()
        for i in range(200):
            target=GOALS[g]-xy
            angular=(np.arctan2(target[1],target[0])-h+np.pi)%(2*np.pi)-np.pi
            h+=np.clip(angular,-.25,.25)
            if abs(angular)<.5:xy+=.018*np.array([np.cos(h),np.sin(h)])
            if np.any((xy<0)|(xy>1)):return -1,i+1
            zones=np.flatnonzero(np.linalg.norm(GOALS-xy,axis=1)<=.06)
            if len(zones):return int(zones[0]),i+1
        return -1,200
    a=[run(p,h,0) for p,h in zip(starts,heading)]
    b=[run(p,h,1) for p,h in zip(starts,heading)]
    both=sum(x[0]==0 and y[0]==1 for x,y in zip(a,b))
    assert both==32
    # Same visits and opportunities, two opposite scalar teaching histories.
    visit_ids=np.tile([0,1,1,0],8)
    rewards=(visit_ids==0).astype(int)
    sham=rng.permutation(rewards)
    assert rewards.sum()==(1-rewards).sum()==sham.sum()
    return {"analytic_paired_first_entries":both,"n":32,"max_steps":max(t for _,t in a+b),
            "fixed_zone_paired":0,"training_visit_separation":float(np.min(np.linalg.norm(starts[:,None]-visits[None],axis=2))),
            "reward_opportunities":len(rewards),"each_history_rewards":int(rewards.sum()),
            "engineering_only":True}


def rewire(u,v,sign,roles):
    rng=np.random.default_rng(SEED)
    groups=defaultdict(list)
    for i,(a,b,s) in enumerate(zip(u,v,sign)):
        groups[(roles[a],roles[b],s)].append(i)
    q=v.copy(); present=set(zip(u.tolist(),v.tolist()));accepted=0
    for ids in groups.values():
        if len(ids)<2:continue
        for i,j in rng.choice(ids,size=(min(2*len(ids),6000),2),replace=True):
            a,b,c,d=int(u[i]),int(q[i]),int(u[j]),int(q[j])
            if a==c or b==d or a==d or c==b or (a,d) in present or (c,b) in present:continue
            present.remove((a,b));present.remove((c,d))
            present.add((a,d));present.add((c,b))
            q[i],q[j]=d,b;accepted+=1
    for s in (-1,1):
        assert np.array_equal(np.bincount(v[sign==s],minlength=len(roles)),
                              np.bincount(q[sign==s],minlength=len(roles)))
    assert np.array_equal(roles[v],roles[q])
    assert accepted>0
    return q,accepted


def stress(u,v,w,roles):
    start=time.process_time()
    mass=np.bincount(v,weights=abs(w),minlength=len(roles))
    weights=1.2*w/np.maximum(mass[v],1e-12)
    h=np.zeros(len(roles))
    cue=np.flatnonzero(np.isin(roles,["ER","PEN","PFN"]))
    rng=np.random.default_rng(SEED)
    maximum=0
    for step in range(120):
        drive=np.zeros(len(roles))
        if step<40:drive[cue]=rng.uniform(0,.6,len(cue))
        h=.8*h+.2*np.tanh(drive+np.bincount(v,weights=weights*h[u],minlength=len(h)))
        assert np.isfinite(h).all()
        maximum=max(maximum,np.max(abs(h)))
    return {"cpu_seconds":time.process_time()-start,"max_abs_state":float(maximum),
            "late_EPG_norm":float(np.linalg.norm(h[roles=="EPG"]))}


def cycle2(g):
    a=g["usable"]
    u,v,s,w=[g[key][a] for key in ("u","v","sign","w")]
    roles=g["roles"]
    assert len(set(zip(u.tolist(),v.tolist())))==len(u)
    q,accepted=rewire(u,v,s,roles)
    h,p=trajectory(0,[(np.pi/2,0),(0,1)])
    _,r=trajectory(0,[(0,1),(np.pi/2,0)])
    assert np.allclose(p,[0,1]) and np.allclose(r,[1,0])
    return {"arena":arena(),"known_angle_path_order_canary":True,
            "rewired_swaps":accepted,"changed_edges":int((v!=q).sum()),
            "role_pairs_and_signed_degrees_preserved":True,
            "numerical_rollout":{"native":stress(u,v,s*w,roles),"rewired":stress(u,q,s*w,roles)},
            "anatomical_heading_and_displacement":"blocked before functional scoring by missing compartment mapping and boundary adapters"}


class Local:
    def __init__(self,g):
        self.n=len(g["roles"])
        u,v=g["u"],g["v"]
        r=g["roles"]
        a=(r[u]=="PN")&(r[v]=="KC")&g["usable"]
        self.pu,self.pv,self.pw=u[a],v[a],g["w"][a]
        self.norm=np.bincount(self.pv,weights=self.pw,minlength=self.n).clip(1e-9)
        mask=g["plastic"]&g["usable"]
        self.u,self.v,self.base=u[mask],v[mask],g["w"][mask]
        addressed=set(v[(r[u]=="DAN")&(r[v]=="KC")])
        self.eligible=np.isin(self.u,list(addressed))
        self.post_norm=np.bincount(self.v,weights=self.base,minlength=self.n).clip(1e-9)
        self.w=self.base.copy()
        self.e=np.zeros(len(self.w))
        self.ema=np.zeros(self.n)
        self.rbar=0.
        self.pns=np.flatnonzero(r=="PN")
        self.mbons=np.flatnonzero(r=="MBON")

    def observe(self,i):
        rng=np.random.default_rng(SEED+i)
        z=np.zeros(self.n)
        z[rng.choice(self.pns,32,replace=False)]=1
        x=np.bincount(self.pv,weights=self.pw*z[self.pu],minlength=self.n)/self.norm
        k=np.flatnonzero(x)
        k=k[np.argsort(-x[k],kind="stable")[:256]]
        xx=np.zeros(self.n);xx[k]=1
        y=np.bincount(self.v,weights=self.w*xx[self.u],minlength=self.n)/self.post_norm
        return xx,y

    def step(self,i,r,writes):
        x,y=self.observe(i)
        self.e=.75*self.e+x[self.u]*(y[self.v]-self.ema[self.v])
        self.ema=.9*self.ema+.1*y
        d=.8*(r-self.rbar)*self.e*self.eligible
        self.rbar=.9*self.rbar+.1*r
        if writes:self.w=np.clip(self.w+d,.5*self.base,1.5*self.base)
        return y[self.mbons]

    def digest(self):
        return sha256(b"".join(x.tobytes() for x in (self.w,self.e,self.ema,np.array(self.rbar)))).hexdigest()

    def save(self,path):
        np.savez(path,w=self.w,e=self.e,ema=self.ema,rbar=self.rbar)

    def load(self,path):
        with np.load(path) as z:
            self.w,self.e,self.ema,self.rbar=(z[k].copy() for k in ("w","e","ema","rbar"))


def cycle3(g,scratch):
    results={}
    for mode in ("paired","shifted","no_write"):
        m=Local(g)
        w0=m.w.copy()
        before=m.digest()
        for i in range(64):
            cue=i%2
            reward=int(cue==(0 if i<32 else 1))
            if mode=="shifted":reward=1-reward
            m.step(cue,reward,mode!="no_write")
            if i in (31,63):
                result=results.setdefault(mode,[])
                result.append({"trial":i+1,"changed_edges":int(np.count_nonzero(m.w!=w0)),
                               "cue_outputs":[m.observe(k)[1][m.mbons].tolist() for k in (0,1)]})
        assert np.all(m.w>0) and not np.any((m.w!=w0)&~m.eligible)
        assert (np.array_equal(m.w,w0))==(mode=="no_write")
        scored=m.digest()
        m.observe(0)
        assert m.digest()==scored and m.digest()!=before
        if mode=="paired":
            path=scratch/"local.npz"
            m.save(path)
            m.step(0,1,True)
            restart={"expected":m.digest(),"checkpoint_sha256":sha256(path.read_bytes()).hexdigest()}
    return {"modes":results,"eligible_DAN_addressed_KC_MBON_edges":int(m.eligible.sum()),
            "restart":restart,"scope":"anatomical-edge odor write fixture; no learned arena policy"}


def main():
    mode=sys.argv[1]
    g=np.load(ROOT/"EXTRACT.npz",allow_pickle=False)
    p=ROOT/("CYCLE2.json" if mode=="2" else "CYCLE3.json")
    if mode=="2":result=cycle2(g)
    else:
        scratch=Path(sys.argv[2])
        if mode=="3":result=cycle3(g,scratch)
        else:
            result=json.loads(p.read_text());m=Local(g);path=scratch/"local.npz"
            assert sha256(path.read_bytes()).hexdigest()==result["restart"]["checkpoint_sha256"]
            m.load(path);m.step(0,1,True)
            result["restart"]["actual"]=m.digest()
            assert m.digest()==result["restart"]["expected"]
    result["status"]="RESTART_PENDING" if mode=="3" else "ENGINEERING_PASS"
    p.write_text(json.dumps(result,indent=2)+"\n")
    print(mode,result["status"],flush=True)


if __name__=="__main__":main()


In [ ]:
%%writefile cx/qualify_interfaces.py
"""Fixed-gain v630 compass and steering probes on development fixtures."""
from hashlib import sha256
import json
from pathlib import Path

import numpy as np

from qualify import rewire

ROOT = Path(__file__).resolve().parent
HEADINGS = 2 * np.pi * (np.arange(16) + .25) / 16
STEP_SECONDS = .05
HOLD_STEPS = 40


def wrap(angle):
    return np.angle(np.exp(1j * angle))


class Interface:
    def __init__(self, graph, coordinates, boundary, synapses):
        labels, self.roles, self.sides = (graph[k] for k in ('labels', 'roles', 'sides'))
        self.epg = self.roles == 'EPG'
        self.pen_a = np.char.startswith(labels, 'PEN_a')
        self.er = labels == 'ER4d'
        self.fc2 = self.roles == 'FC2'
        self.pfl3 = labels == 'PFL3'
        cells = {r['root_id']: r for r in coordinates['cells']}
        ids = graph['ids']
        eb = {row['root_id']: row['EB'] for row in synapses['epg']}
        self.phase = np.deg2rad([eb[k]['phase_deg'] for k in ids[self.epg]])
        # Least-squares spatial first harmonic with a DC column. Its design matrix uses only anatomy.
        self.observer = np.linalg.pinv(np.column_stack((np.ones(self.epg.sum()), np.cos(self.phase), np.sin(self.phase))))[1:]
        pb = np.array([cells[k]['PB']['position_um'] for k in ids[self.epg]])
        pb_position = np.array([cells.get(k, {}).get('PB', {}).get('position_um', np.nan) for k in ids])
        pfl3_pb = np.array([cells[k]['PB']['position_um'] for k in ids[self.pfl3]])
        pfl3_x = np.array([cells[k]['FB']['centroid_um'][0] for k in ids[self.pfl3]])
        fc2_x = np.array([cells[k]['FB']['centroid_um'][0] for k in ids[self.fc2]])
        # One rotation of FB phase across its PFL3-covered x span. Fit a circular offset from
        # PB/FB geometry on each side separately, then take the midpoint of the opponent shifts.
        pfl3_phase = np.angle(np.exp(-.5 * ((pfl3_pb[:, None] - pb) / 3.)**2) @ np.exp(1j * self.phase))
        slope = 2 * np.pi / np.ptp(pfl3_x)
        by_side = self.sides[self.pfl3]
        offsets = [np.mean(np.exp(1j * (pfl3_phase - slope * pfl3_x))[by_side == s]) for s in ('left', 'right')]
        self.fb_phase_slope = slope
        self.fb_phase_offset = np.angle(sum(z / abs(z) for z in offsets))
        self.fc2_phase = wrap(slope * fc2_x + self.fb_phase_offset)
        self.fc2_alignment = dict(spatial_period_um=float(np.ptp(pfl3_x)),
                                  side_alignment_resultants=[float(abs(z)) for z in offsets])

        u, v, w, signs, usable = (graph[k] for k in ('u', 'v', 'w', 'sign', 'usable'))
        er_phase = np.zeros(len(labels), dtype=complex)
        phase_vector = np.zeros(len(labels), dtype=complex)
        phase_vector[self.epg] = np.exp(1j * self.phase)
        for source in np.flatnonzero(self.er):
            a = (u == source) & self.epg[v] & usable
            er_phase[source] = w[a] @ phase_vector[v[a]] / max(w[a].sum(), 1)
        self.er_phase = np.angle(er_phase[self.er])
        self.er_selectivity = abs(er_phase[self.er])
        motor = np.zeros((2, len(labels)))
        idx = {k: i for i, k in enumerate(ids)}
        for row in boundary['pfl3_to_dna02']:
            if row['consistent']:
                side = ('left', 'right').index(row['target_side'])
                motor[side, idx[row['source']]] = row['raw_signed_weight']
        assert np.all(motor.sum(axis=1) > 0) and np.all(np.count_nonzero(motor, axis=0)[self.pfl3] == 1)
        self.motor = motor / motor.sum(axis=1, keepdims=True)
        self.motor_edge_count = int(np.count_nonzero(motor))
        self.glno_current = np.zeros(len(labels))
        for row in boundary['glno_to_pen_a']:
            if row['consistent']:
                source_sign = 1. if row['source_side'] == 'left' else -1.
                self.glno_current[idx[row['target']]] += source_sign * row['raw_signed_weight']
        self.glno_current /= np.median(np.abs(self.glno_current[self.pen_a]))
        assert np.isfinite(self.glno_current).all()

        core = self.epg | self.er | self.fc2 | self.pfl3 | np.isin(self.roles, ['D7', 'PEG', 'PEN'])
        graph_scope = core[u] & core[v]
        sensitivity = np.isin(ids[u], boundary['glutamatergic_delta7']) & (self.epg[v] | self.pfl3[v])
        assert not np.any(usable & sensitivity)
        self.compass = np.isin(self.roles, ['EPG', 'PEN', 'PEG', 'D7'])
        self.circuits = {}
        self.variants = {method: {} for method in ('aggregate', 'pb_proximity', 'observed_contacts')}
        for name, mask in [('native', usable), ('delta7_inhibitory', usable | sensitivity)]:
            sel = graph_scope & mask
            sign = np.where(sensitivity[sel], -1, signs[sel])
            a, b, weight = u[sel], v[sel], w[sel] * sign
            # Rewire within anatomical subtype and side, retaining the same boundary neurons.
            null_labels = np.char.add(np.char.add(labels, '/'), self.sides)
            null, swaps = rewire(a, b, sign, null_labels)
            self.circuits[name] = self.normalize(a, b, weight)
            self.circuits[name + '_rewired'] = self.normalize(a, null, weight)
            self.circuits[name + '_rewired_swaps'] = dict(accepted=swaps, changed=int(np.count_nonzero(b != null)))
            if name == 'delta7_inhibitory':
                filtered = weight.astype(float)
                epg_to_pen = self.epg[a] & self.pen_a[b]
                filtered[epg_to_pen] *= np.exp(-.5 * ((pb_position[a[epg_to_pen]] - pb_position[b[epg_to_pen]]) / 3.)**2)
                self.pen_pb_phase_shift = self.pen_offsets(a, b, weight, filtered)
                for method, effective in [('aggregate', weight), ('pb_proximity', filtered)]:
                    self.variants[method] = {'': (a, b, effective), '_rewired': (a, null, effective)}
                self.register_observed_compartments(a, b, weight, sign, ids, synapses, null_labels)
                epg_excit = self.epg[b] & self.compass[a] & (weight > 0)
                epg = self.epg[b]
                fractions = (np.bincount(b[epg_excit], weights=weight[epg_excit], minlength=len(labels)) /
                             np.bincount(b[epg], weights=abs(weight[epg]), minlength=len(labels)).clip(1e-12))
                self.epg_excitatory_fraction = float(np.median(fractions[self.epg]))
        # Removing the compass recurrent inputs keeps cue and yaw drives identical across arms.
        a, b, w = self.circuits['native']
        keep = ~(self.compass[a] & self.compass[b])
        self.circuits['recurrence_off'] = a[keep], b[keep], w[keep]

    def normalize(self, source, target, w):
        mass = np.bincount(target, weights=abs(w), minlength=len(self.roles))
        return source, target, 1.2 * w / np.maximum(mass[target], 1e-12)

    def register_observed_compartments(self, a, b, weight, sign, ids, synapses, null_labels):
        sites = {(row['source'], row['target']): row for row in synapses['epg_pen_a_connections']}
        exact = weight.astype(float)
        afferent, efferent = self.epg[a] & self.pen_a[b], self.pen_a[a] & self.epg[b]
        stats = {}
        for route, roi in [(afferent, 'pb'), (efferent, 'eb')]:
            indices = np.flatnonzero(route)
            found = 0
            for i in indices:
                row = sites.get((ids[a[i]], ids[b[i]]))
                if row is not None:
                    found += 1
                    exact[i] *= row[roi] / row['total']
                else:
                    exact[i] = 0
            stats[roi] = dict(aggregate_edges=len(indices), coordinate_export_pairs=found,
                              edges_with_contacts_in_compartment=int(np.count_nonzero(exact[indices])))
        keep = exact != 0
        q, swaps = rewire(a[keep], b[keep], sign[keep], null_labels)
        self.variants['observed_contacts'] = {'': (a[keep], b[keep], exact[keep]), '_rewired': (a[keep], q, exact[keep])}
        self.observed_compartments = dict(stats, rewired_swaps=swaps, changed=int(np.count_nonzero(b[keep] != q)),
                                          median_feedback_phase_shift_deg=self.pen_offsets(a, b, weight, exact, exact))

    def pen_offsets(self, source, target, weights, filtered, filtered_outputs=None):
        phases = np.zeros(len(self.roles), dtype=complex)
        phases[self.epg] = np.exp(1j * self.phase)
        offsets = {'left': [[], []], 'right': [[], []]}
        if filtered_outputs is None:
            filtered_outputs = weights
        for i in np.flatnonzero(self.pen_a):
            entering = (target == i) & self.epg[source]
            exiting = (source == i) & self.epg[target]
            for result, w_in, w_out in zip(offsets[self.sides[i]], (weights, filtered), (weights, filtered_outputs)):
                in_phase = w_in[entering] @ phases[source[entering]]
                out_phase = w_out[exiting] @ phases[target[exiting]]
                result.append(np.rad2deg(np.angle(out_phase * in_phase.conjugate())))
        return {s: dict(aggregate=float(np.median(raw)), compartment_weighted=float(np.median(pb)))
                for s, (raw, pb) in offsets.items()}

    def separate_compass_gains(self, excitation, inhibition, suffix, method='aggregate'):
        source, target, weight = self.variants[method][suffix]
        a, b, w = self.normalize(source, target, weight)
        recurrent = self.compass[source] & self.compass[target]
        landmark = self.er[source] & self.epg[target]
        # An inactive sensory pathway should not dilute compass recurrence. Keep the observed
        # ratios within each of the three independently normalized incoming route classes.
        for mask, gain in [(recurrent & (weight > 0), excitation), (recurrent & (weight < 0), inhibition),
                           (landmark, .4)]:
            mass = np.bincount(target[mask], weights=abs(weight[mask]), minlength=len(self.roles))
            w[mask] = gain * weight[mask] / mass[target[mask]]
        self.circuits['separated' + suffix] = a, b, w

    def initial_state(self):
        return np.zeros(len(self.roles))

    def step(self, state, circuit, landmark_bearing=None, yaw=0., supplied_goal=None, sensory='epg',
             yaw_scale=.06, yaw_adapter='pen_gain'):
        a, b, w = self.circuits[circuit]
        current = np.bincount(b, weights=w * state[a], minlength=len(state))
        if landmark_bearing is not None:
            # The fixture has one landmark at phase zero and supplies its egocentric bearing.
            if sensory == 'epg':
                current[self.epg] += .9 * np.exp(4 * (np.cos(self.phase + landmark_bearing) - 1))
            elif sensory in ('er', 'er_constant'):
                contrast = self.er_selectivity * np.cos(self.er_phase + landmark_bearing) if sensory == 'er' else 0.
                current[self.er] += .8 * (1 - contrast)
                # Inhibition can modulate an active compass. It cannot seed activity from rest.
                current[self.epg] += .65
            else:
                raise ValueError(f'Unknown sensory adapter {sensory}')
        # Engineered velocity gain at PEN_a. Positive yaw is a left turn in the fixture.
        if yaw_adapter == 'glno':
            # A centered perturbation around tonic GLNO activity. Its assumed opponent sensory
            # drive is held fixed across graphs, with the signed v630 GLNO→PEN1 weights.
            current += np.clip(yaw / yaw_scale, -1, 1) * self.glno_current
        else:
            pen_side = np.where(self.sides[self.pen_a] == 'left', 1., -1.)
            current[self.pen_a] *= 1 + np.maximum(0, yaw * pen_side / yaw_scale)
        if supplied_goal is not None:
            current[self.fc2] += .9 * np.exp(4 * (np.cos(self.fc2_phase - supplied_goal) - 1))
        return .8 * state + .2 * np.tanh(np.maximum(0, current))

    def heading(self, state):
        x, y = self.observer @ state[self.epg]
        return np.arctan2(y, x), np.hypot(x, y)

    def steer(self, state):
        left, right = self.motor @ state
        return left - right


def compass_probe(model, circuit, sensory, headings=HEADINGS, yaw_scale=.06, yaw_adapter='pen_gain'):
    visible, stationary, turning = [], [], []
    for angle in headings:
        initial = model.initial_state()
        for _ in range(HOLD_STEPS):
            initial = model.step(initial, circuit, landmark_bearing=-angle, sensory=sensory)
        locked, strength = model.heading(initial)
        visible.append((abs(wrap(locked - angle)), strength))
        stationary_phase = None
        for yaw in (0., -.04, .04):
            state = initial.copy()
            for _ in range(HOLD_STEPS):
                state = model.step(state, circuit, yaw=yaw, yaw_scale=yaw_scale, yaw_adapter=yaw_adapter)
            phase, remaining = model.heading(state)
            retention = remaining / max(strength, 1e-12)
            if yaw == 0:
                stationary_phase = phase
                stationary.append((abs(wrap(phase - locked)), retention))
            else:
                turning.append((abs(wrap(phase - angle - HOLD_STEPS * yaw)), retention,
                                np.sign(yaw) * wrap(phase - stationary_phase)))
    visible, stationary, turning = map(np.asarray, (visible, stationary, turning))
    return dict(landmark_median_error_deg=float(np.rad2deg(np.median(visible[:, 0]))),
                landmark_median_harmonic=float(np.median(visible[:, 1])),
                landmark_n_with_harmonic_at_least_0_02=int(np.count_nonzero(visible[:, 1] >= .02)),
                stationary_with_at_least_quarter_strength=int(np.count_nonzero(stationary[:, 1] >= .25)),
                stationary_with_quarter_strength_and_drift_at_most_20_deg=int(np.count_nonzero(
                    (stationary[:, 1] >= .25) & (stationary[:, 0] <= np.deg2rad(20)))),
                stationary_n=len(stationary), stationary_median_strength_ratio=float(np.median(stationary[:, 1])),
                turn_median_error_deg=float(np.rad2deg(np.median(turning[:, 0]))),
                turn_median_directional_response_deg=float(np.rad2deg(np.median(turning[:, 2]))),
                turn_median_strength_ratio=float(np.median(turning[:, 1])), turn_n=len(turning))


def steering_probe(model, circuit):
    rows = []
    for heading in HEADINGS:
        for goal_offset in (-np.pi / 2, -np.pi / 4, np.pi / 4, np.pi / 2):
            state = model.initial_state()
            no_goal = model.initial_state()
            for _ in range(HOLD_STEPS):
                state = model.step(state, circuit, landmark_bearing=-heading, supplied_goal=heading + goal_offset)
                no_goal = model.step(no_goal, circuit, landmark_bearing=-heading)
            rows.append((goal_offset, model.steer(state), model.steer(no_goal)))
    target, supplied, absent = np.asarray(rows).T
    return dict(correct_turn_sign=int(np.count_nonzero(target * supplied > 0)),
                no_goal_correct_turn_sign=int(np.count_nonzero(target * absent > 0)),
                goal_effect_correct_turn_sign=int(np.count_nonzero(target * (supplied - absent) > 0)),
                n=len(rows), median_abs_goal_effect=float(np.median(abs(supplied - absent))))


def gain_probe(model):
    # A bounded development grid. Probe a different angular panel from the original smoke test.
    headings = HEADINGS + np.pi / 16
    trials = []
    for excite in (1.2, 1.8, 2.4):
        for inhibit in (.4, .8, 1.2):
            for suffix in ('', '_rewired'):
                model.separate_compass_gains(excite, inhibit, suffix)
            trials.append(dict(excitatory_gain=excite, inhibitory_gain=inhibit,
                               native=compass_probe(model, 'separated', 'epg', headings),
                               rewired=compass_probe(model, 'separated_rewired', 'epg', headings)))
    return trials


def compartment_probe(model, method):
    # Follow-up on a shifted panel after the separated-gain development grid.
    headings = HEADINGS + np.pi / 32
    trials = []
    for inhibition in (.4, .8):
        for suffix in ('', '_rewired'):
            model.separate_compass_gains(1.2, inhibition, suffix, method=method)
        for yaw_scale in (.02, .04, .06):
            trials.append(dict(excitatory_gain=1.2, inhibitory_gain=inhibition, yaw_scale=yaw_scale,
                               native=compass_probe(model, 'separated', 'epg', headings, yaw_scale),
                               rewired=compass_probe(model, 'separated_rewired', 'epg', headings, yaw_scale)))
    return trials


def glno_probe(model):
    headings = HEADINGS + 3 * np.pi / 64
    for suffix in ('', '_rewired'):
        model.separate_compass_gains(1.2, .8, suffix, method='observed_contacts')
    return [dict(yaw_scale=scale,
                 native=compass_probe(model, 'separated', 'epg', headings, scale, yaw_adapter='glno'),
                 rewired=compass_probe(model, 'separated_rewired', 'epg', headings, scale, yaw_adapter='glno'))
            for scale in (.04, .08, .16)]


def velocity_probe(model, graph, coordinates):
    """Test instantaneous velocity with a clamped compass and no displacement state."""
    roles, labels, sides, ids = (graph[k] for k in ('roles', 'labels', 'sides', 'ids'))
    pfnd, pfnv, hdelta, pfr = (labels == t for t in ('PFNd', 'PFNv', 'hDeltaB', 'PFR'))
    pfn = pfnd | pfnv
    cells = {row['root_id']: row for row in coordinates['cells']}
    pfr_x = np.array([cells[k]['FB']['centroid_um'][0] for k in ids[pfr]])
    phase = model.fb_phase_slope * pfr_x + model.fb_phase_offset
    observer = np.linalg.pinv(np.column_stack((np.ones(pfr.sum()), np.cos(phase), np.sin(phase))))[1:]
    # Forward positive, left positive. The four body axes follow Lyu et al. Fig. 4.
    left = sides[pfn] == 'left'
    axes = np.where(pfnd[pfn], np.where(left, 45., -45.), np.where(left, -135., 135.))
    axes = np.deg2rad(axes)
    u, v, w, sign, usable = (graph[k] for k in ('u', 'v', 'w', 'sign', 'usable'))
    route = (model.epg[u] & pfn[v]) | (pfn[u] & hdelta[v]) | (hdelta[u] & pfr[v])
    a, b, w = u[route & usable], v[route & usable], w[route & usable] * sign[route & usable]
    q, swaps = rewire(a, b, np.sign(w), np.char.add(np.char.add(labels, '/'), sides))
    n = len(roles)
    report = {}
    directions = np.pi * np.arange(8) / 4
    for name, target in [('native', b), ('rewired', q)]:
        mass = np.bincount(target, weights=np.abs(w), minlength=n).clip(1e-12)

        def current(source):
            return np.bincount(target, weights=w * source[a], minlength=n) / mass

        errors, strength, zero = [], [], []
        for heading in HEADINGS:
            epg = np.zeros(n)
            epg[model.epg] = np.exp(4 * (np.cos(model.phase - heading) - 1))
            heading_current = current(epg)[pfn]
            for direction in np.r_[directions, np.nan]:
                pfn_state = np.zeros(n)
                if np.isfinite(direction):
                    pfn_state[pfn] = heading_current * np.maximum(0, np.cos(direction - axes))
                hdelta_state = current(pfn_state) * hdelta
                x, y = observer @ current(hdelta_state)[pfr]
                if np.isfinite(direction):
                    errors.append(abs(wrap(np.arctan2(y, x) - heading - direction)))
                    strength.append(np.hypot(x, y))
                else:
                    zero.append(np.hypot(x, y))
        report[name] = dict(n=len(errors), median_direction_error_deg=float(np.rad2deg(np.median(errors))),
                            median_harmonic=float(np.median(strength)), max_zero_motion_harmonic=float(max(zero)))
    return dict(report, changed_edges=int(np.count_nonzero(b != q)), accepted_rewire_swaps=swaps,
                description='Feedforward EPG→PFNd/v→hDeltaB→PFR with externally clamped compass and no integrated position')


def main():
    names = ['EXTRACT.npz', 'COMPARTMENT_MAP.json', 'RESOLVED_BOUNDARY.json', 'SYNAPSE_COMPARTMENTS.json']
    extract, map_file, boundary_file, synapse_file = (ROOT / name for name in names)
    with np.load(extract, allow_pickle=False) as source:
        graph = dict(source)
    coordinates = json.loads(map_file.read_text())
    model = Interface(graph, coordinates, json.loads(boundary_file.read_text()), json.loads(synapse_file.read_text()))
    # The motor signs are fixed by the anatomical output sides and the published ipsiversive convention.
    for side, expected_sign in [('left', 1), ('right', -1)]:
        pulse = (model.motor[('left', 'right').index(side)] > 0).astype(float)
        assert np.isclose(model.steer(pulse), expected_sign)
    arms = ('native', 'native_rewired', 'recurrence_off', 'delta7_inhibitory', 'delta7_inhibitory_rewired')
    report = dict(fixture='fixed-gain engineering development only', step_seconds=STEP_SECONDS, fitted_scalars=0,
                  graph_inputs={name: sha256((ROOT / name).read_bytes()).hexdigest() for name in names},
                  dna02_motor_edge_count=model.motor_edge_count, dna02_left_right_sign_canary=True,
                  epg_median_compass_excitation_fraction_of_incoming_absolute_mass=model.epg_excitatory_fraction,
                  fc2_geometry=model.fc2_alignment,
                  swaps={k: v for k, v in model.circuits.items() if k.endswith('_swaps')},
                  heading={arm: compass_probe(model, arm, 'epg') for arm in arms},
                  er4d_only={arm: compass_probe(model, arm, 'er') for arm in ('native', 'native_rewired')},
                  er4d_constant_drive_control=compass_probe(model, 'native', 'er_constant'),
                  steering={arm: steering_probe(model, arm) for arm in arms if arm != 'recurrence_off'},
                  separated_compass_gain_development_grid=gain_probe(model),
                  pen1_median_feedback_phase_shift_deg=model.pen_pb_phase_shift,
                  pb_weighted_compass_development_grid=compartment_probe(model, 'pb_proximity'),
                  observed_compartments=model.observed_compartments,
                  observed_compartments_development_grid=compartment_probe(model, 'observed_contacts'),
                  glno_yaw_development_grid=glno_probe(model),
                  clamped_heading_velocity=velocity_probe(model, graph, coordinates),
                  displacement='unopened pending compass qualification', learned_navigation='unopened')
    (ROOT / 'INTERFACE_PROBE.json').write_text(json.dumps(report, indent=2) + '\n')
    print(json.dumps({key: report[key] for key in ('heading', 'er4d_only', 'steering')}, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile cx/qualify_dynamics.py
"""Calibrate explicit compass and steering subcircuits, then qualify on fresh panels."""
from hashlib import sha256
from itertools import product
import json
import numpy as np
from qualify_interfaces import Interface, ROOT, wrap

FILES = ('EXTRACT.npz', 'COMPARTMENT_MAP.json', 'RESOLVED_BOUNDARY.json', 'SYNAPSE_COMPARTMENTS.json')
N = 40
PANEL = np.arange(16) * 2 * np.pi / 16


def bump(phases, angles):
    return np.exp(4 * (np.cos(np.asarray(angles)[..., None] - phases) - 1))


def projection(source, target, weights, pre, post, normalize=True):
    a = np.flatnonzero(pre)
    b = np.flatnonzero(post)
    mask = pre[source] & post[target]
    matrix = np.zeros((len(a), len(b)))
    np.add.at(matrix, (np.searchsorted(a, source[mask]), np.searchsorted(b, target[mask])), weights[mask])
    if normalize:
        matrix /= np.abs(matrix).sum(axis=0).clip(1e-12)
    return matrix


class Compass:
    def __init__(self, model, route, arm):
        a, b, w = model.variants[route]['_rewired' if arm == 'rewired' else '']
        e, p = model.epg, model.pen_a
        d = model.roles == 'D7'
        self.local = projection(a, b, w, e, e)
        self.enter = projection(a, b, w, e, p)
        # Normalize by the combined population, so the two PEN branches compete.
        self.exit = projection(a, b, w, p, e)
        self.inhib = projection(a, b, w, e, d) @ projection(a, b, w, d, e)
        self.side = np.where(model.sides[p] == 'left', 1., -1.)
        self.model = model
        if arm == 'recurrence_off':
            self.local *= 0
            self.inhib *= 0
            self.exit *= 0

    def step(self, e, yaw, params, drive_mode, rest_rate):
        local, inhibit, feedback, yaw_gain = params
        p = np.maximum(e @ self.enter, 0.)
        modulation = np.clip(yaw[:, None] * yaw_gain * self.side, -1., 1.)
        if drive_mode == 'single_side':
            modulation = np.maximum(modulation, 0.)
        resting = local * (e @ self.local) + feedback * (p @ self.exit) + inhibit * (e @ self.inhib)
        active = np.tanh(np.maximum(resting + feedback * ((p * modulation) @ self.exit), 0.))
        if rest_rate == .45:
            return .55 * e + .45 * active
        resting = np.tanh(np.maximum(resting, 0.))
        return np.clip(e + rest_rate * (resting - e) + .45 * (active - resting), 0., 1.)

    def run(self, angles, yaw, params, drive_mode='opponent', rest_rate=.45):
        e = bump(self.model.phase, angles)
        initial = e
        for dt in yaw.T:
            e = self.step(e, dt, params, drive_mode, rest_rate)
        xy = e @ self.model.observer.T
        baseline = np.linalg.norm(initial @ self.model.observer.T, axis=1)
        return np.arctan2(xy[:, 1], xy[:, 0]), np.linalg.norm(xy, axis=1) / baseline


def turn_panel(angles, sequences):
    return np.tile(angles, len(sequences)), np.repeat(sequences, len(angles), axis=0)


def compass_score(compass, angles, yaw, params, mode, rest_rate=.45):
    phase, retention = compass.run(angles, yaw, params, mode, rest_rate)
    moving = np.any(yaw != 0, axis=1)
    error = np.rad2deg(abs(wrap(phase - angles - yaw.sum(axis=1))))
    passes = (retention >= .25) & (error <= 20.)
    return dict(stationary_pass=int(passes[~moving].sum()), stationary_n=int((~moving).sum()),
                turn_median_deg=float(np.median(error[moving])), turn_n=int(moving.sum()),
                turn_pass=int(passes[moving].sum()),
                turn_retained=int((retention[moving] >= .25).sum()),
                stationary_median_deg=float(np.median(error[~moving])))


def compass_experiment(model):
    dev = turn_panel(PANEL + .04, [np.zeros(N), np.full(N, .04), np.full(N, -.04)])
    held = turn_panel(PANEL + .19, [np.zeros(N), np.full(N, .03), np.full(N, -.03),
                                    np.r_[np.full(10, .065), np.full(30, -.009)],
                                    np.r_[np.full(10, -.065), np.full(30, .009)],
                                    np.r_[np.zeros(16), np.full(24, .06)],
                                    np.r_[np.zeros(16), np.full(24, -.06)]])
    report = {}
    for route in ('aggregate', 'pb_proximity', 'observed_contacts'):
        report[route] = {}
        for arm in ('native', 'rewired'):
            compass = Compass(model, route, arm)
            candidates = []
            for params in product((.6, .95, 1.25, 1.55), (.4, .8, 1.2), (.3, .65, 1.), (6., 15., 30., 55.)):
                for mode in ('opponent', 'single_side'):
                    row = compass_score(compass, *dev, params, mode)
                    # Penalize compass collapse and score angle only after stationary coverage.
                    key = (-min(row['stationary_pass'], 13), -row['turn_pass'], row['turn_median_deg'])
                    candidates.append((key, params, mode, row))
            _, params, mode, calibration = min(candidates, key=lambda row: row[0])
            qualification = compass_score(compass, *held, params, mode)
            report[route][arm] = dict(params=params, yaw_mode=mode, calibration=calibration, qualification=qualification)
            if arm == 'native':
                for lesion in ('recurrence_off', 'rewired'):
                    report[route][lesion + '_native_parameters'] = compass_score(
                        Compass(model, route, lesion), *held, params, mode)
                report[route]['no_yaw_native_parameters'] = compass_score(compass, *held, (*params[:3], 0.), mode)
    return report


def separate_timescale_experiment(model):
    dev = turn_panel(PANEL + .07, [np.zeros(N), np.full(N, .04), np.full(N, -.04),
                                   np.r_[np.full(18, .055), np.full(22, -.017)],
                                   np.r_[np.full(18, -.055), np.full(22, .017)]])
    # Fresh headings and yaw schedules for the follow-up after the first panel was examined.
    held = turn_panel(PANEL + .31, [np.zeros(N), np.full(N, .027), np.full(N, -.027),
                                    np.r_[np.full(17, .054), np.full(23, -.014)],
                                    np.r_[np.full(17, -.054), np.full(23, .014)],
                                    np.r_[np.zeros(12), np.full(28, .054)],
                                    np.r_[np.zeros(12), np.full(28, -.054)]])
    report = {}
    for route in ('pb_proximity', 'observed_contacts'):
        results = report[route] = {}
        for arm in ('native', 'rewired'):
            compass = Compass(model, route, arm)
            best = None
            for rest, local, inhibition, feedback, gain in product(
                (.055, .085, .13), (.75, 1.25, 1.55), (.4, .8), (.3, .65, 1.), (8., 16., 30.)
            ):
                params = (local, inhibition, feedback, gain)
                row = compass_score(compass, *dev, params, 'opponent', rest)
                key = (-min(row['stationary_pass'], 13), -row['turn_pass'], row['turn_median_deg'])
                if best is None or key < best[0]:
                    best = (key, params, rest, row)
            _, params, rest, row = best
            results[arm] = dict(params=params, rest_rate=rest, calibration=row,
                                qualification=compass_score(compass, *held, params, 'opponent', rest))
            if arm == 'native':
                for control in ('rewired', 'recurrence_off', 'no_yaw'):
                    net = compass if control == 'no_yaw' else Compass(model, route, control)
                    p = (*params[:3], 0.) if control == 'no_yaw' else params
                    results[control + '_native_parameters'] = compass_score(net, *held, p, 'opponent', rest)
    return report


class Steering:
    def __init__(self, model, arm):
        a, b, w = model.variants['aggregate']['_rewired' if arm == 'rewired' else '']
        e, f, p = model.epg, model.fc2, model.pfl3
        d = model.roles == 'D7'
        self.head = projection(a, b, w, e, p)
        self.delta = projection(a, b, w, e, d) @ projection(a, b, w, d, p)
        self.goal = projection(a, b, w, f, p)
        self.motor = model.motor[:, p]
        self.model = model
        # Center and scale the two input streams independently. Normalization uses
        # a uniform phase panel and no turn outcomes or target offsets.
        ph = np.arange(64) * (2 * np.pi / 64)
        self.stats = [[(x.mean(axis=0), x.std(axis=0).clip(.04)) for x in self.streams(ph, ph, weight)]
                      for weight in (0., .5, 1.)]

    def streams(self, headings, goals, weight):
        state = bump(self.model.phase, headings)
        return state @ (self.head + weight * self.delta), bump(self.model.fc2_phase, goals) @ self.goal

    def command(self, headings, goals, params, lesion=None):
        offset, weight, goal_gain, threshold = params
        h, g = self.streams(headings, goals + offset, weight)
        (h_mean, h_scale), (g_mean, g_scale) = self.stats[int(weight * 2)]
        h, g = (h - h_mean) / h_scale, (g - g_mean) / g_scale
        if lesion == 'no_goal':
            g *= 0.
        if lesion == 'no_heading':
            h *= 0.
        response = np.logaddexp(0., 3 * (h + goal_gain * g - threshold)) / 3
        return response @ (self.motor[0] - self.motor[1])


def steering_panel(headings, offsets):
    h = np.repeat(headings, len(offsets))
    return h, h + np.tile(offsets, len(headings))


def steering_score(steering, headings, goals, params, lesion=None, closed=True):
    truth = wrap(goals - headings)
    u = steering.command(headings, goals, params, lesion)
    correct = int(np.count_nonzero(u * truth > 0.))
    pair_correct = int(np.all((u * truth > 0).reshape(-1, 2), axis=1).sum())
    result = dict(correct=correct, n=len(u), opposite_goal_pairs=pair_correct,
                  median_abs_output=float(np.median(abs(u))))
    if closed:
        state = headings.copy()
        for _ in range(80):
            command = steering.command(state, goals, params, lesion)
            state += np.clip(.14 * command, -.12, .12)
        error = np.rad2deg(abs(wrap(goals - state)))
        result.update(closed_loop_median_error_deg=float(np.median(error)),
                      closed_loop_within_20_deg=int((error <= 20.).sum()))
    return result


def steering_experiment(model):
    dev = steering_panel(PANEL + .04, np.deg2rad([-40, 40, -95, 95]))
    held = steering_panel(PANEL + .19, np.deg2rad([-30, 30, -65, 65, -120, 120]))
    report = {}
    for arm in ('native', 'rewired'):
        s = Steering(model, arm)
        rows = []
        # Motor polarity stays fixed to DNa02 laterality. The phase offset calibrates
        # the FB goal coordinate into the EPG coordinate on the development panel.
        for params in product(np.arange(24) * np.pi / 12, (0., .5, 1.), (.5, 1., 2.), (.25, .75, 1.25)):
            row = steering_score(s, *dev, params, closed=False)
            if row['correct'] >= 42:
                closed = steering_score(s, *dev, params)
                rows.append(((-closed['closed_loop_within_20_deg'], -row['opposite_goal_pairs'],
                              -row['correct'], closed['closed_loop_median_error_deg']), params, closed))
            else:
                rows.append(((0, -row['opposite_goal_pairs'], -row['correct'], 180.), params, row))
        _, params, calibration = min(rows, key=lambda row: row[0])
        report[arm] = dict(params=params, calibration=calibration,
                           qualification=steering_score(s, *held, params),
                           no_goal=steering_score(s, *held, params, 'no_goal'),
                           no_heading=steering_score(s, *held, params, 'no_heading'))
        if arm == 'native':
            report['rewired_native_parameters'] = steering_score(Steering(model, 'rewired'), *held, params)
    return report


def combined_probe(model, turn, steering, starts, goals, yaw_limit):
    """Close the loop using decoded compass state after a single initial cue."""
    report = {}
    for arm in ('native', 'no_yaw', 'no_goal', 'rewired'):
        graph_arm = 'rewired' if arm == 'rewired' else 'native'
        compass = Compass(model, 'observed_contacts', graph_arm)
        controller = Steering(model, graph_arm)
        turn_params = turn[graph_arm]['params']
        steering_params = steering[graph_arm]['params']
        if arm == 'no_yaw':
            turn_params = (*turn_params[:3], 0.)
        e = bump(model.phase, starts)
        position = starts.copy()  # Simulator heading, used for scoring only after the initial cue.
        panel = {}
        for step in range(1, 81):
            x, y = (e @ model.observer.T).T
            heading = np.arctan2(y, x)
            action = controller.command(heading, goals, steering_params, 'no_goal' if arm == 'no_goal' else None)
            yaw = np.clip(.14 * action, -yaw_limit, yaw_limit)
            position += yaw
            e = compass.step(e, yaw, turn_params, 'opponent', turn[graph_arm]['rest_rate'])
            if step in (40, 80):
                error = np.rad2deg(abs(wrap(goals - position)))
                x, y = (e @ model.observer.T).T
                compass_error = np.rad2deg(abs(wrap(np.arctan2(y, x) - position)))
                panel[str(step)] = dict(n=len(starts), within_20_deg=int((error <= 20.).sum()),
                                        goal_error_median_deg=float(np.median(error)),
                                        compass_error_median_deg=float(np.median(compass_error)))
        report[arm] = panel
    return report


def main():
    with np.load(ROOT / FILES[0], allow_pickle=False) as source:
        graph = dict(source)
    model = Interface(graph, *(json.loads((ROOT / name).read_text()) for name in FILES[1:]))
    turn = separate_timescale_experiment(model)
    steering = steering_experiment(model)
    initial_combined = steering_panel(PANEL + .47, np.deg2rad([-42, 42, -80, 80, -110, 110]))
    rate_panel = steering_panel(PANEL + .59, np.deg2rad([-36, 36, -71, 71, -105, 105]))
    report = dict(graph_inputs={name: sha256((ROOT / name).read_bytes()).hexdigest() for name in FILES},
                  parameter_order=dict(compass=['EPG recurrence', 'Delta7 inhibition', 'PEN feedback', 'yaw gain'],
                                       steering=['FB phase offset in radians', 'Delta7 gain', 'FC2 gain', 'PFL3 threshold']),
                  compass=compass_experiment(model), steering=steering,
                  compass_separate_timescales=turn,
                  combined_supplied_goal=combined_probe(model, turn['observed_contacts'], steering, *initial_combined, .12),
                  combined_rate_check={str(limit): combined_probe(model, turn['observed_contacts'], steering, *rate_panel, limit)
                                       for limit in (.054, .12)},
                  scope='Engineering calibration and disjoint qualification panels. No learned-goal or displacement run.')
    (ROOT / 'DYNAMICS_QUALIFICATION.json').write_text(json.dumps(report, indent=2) + '\n')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile blocks.py
"""Portable pilot and development runners for the MiniFly sweep."""
from itertools import islice
import json
from pathlib import Path
import sys

import numpy as np

ROOT = Path(__file__).resolve().parent


def panel_hash(values):
    from hashlib import sha256
    return sha256(json.dumps(values, separators=(',', ':')).encode()).hexdigest()


def cortical(arm, seed, profile):
    import mod97_local as m

    dose, pair_count = {'smoke': (8, 8), 'pilot': (512, 32), 'screen': (12288, 64)}[profile]
    fit, monitor, held = m.panels(seed)
    canonical = lambda rows: {tuple(sorted(row)) for row in rows}
    assert not canonical(fit) & canonical(held)
    assert not canonical(monitor) & canonical(held)
    groups = np.asarray(sorted({(a, b) for a, b in held if a < b}))
    rng = np.random.default_rng(seed + 106)
    wrap_groups = groups[groups.sum(axis=1) >= 97]
    ordinary_groups = groups[groups.sum(axis=1) < 97]
    pair_set = np.concatenate([g[rng.choice(len(g), pair_count // 2, replace=False)]
                               for g in (wrap_groups, ordinary_groups)])
    pairs = [(int(a), int(b)) for a, b in pair_set]
    net, feedback, optimizer = m.make(seed, arm)
    update = m.update_bp if arm == 'bp' else m.update_dfa
    loss = [update(net, feedback, optimizer, rows) for rows in islice(m.stream(fit, seed), dose)]
    prompts = [m.arithmetic(p) for a, b in pairs for p in ((a, b), (b, a))]
    net.eval()
    out = m.respond(net, [row[:6] for row in prompts])
    correct = [actual == row[6:] for actual, row in zip(out, prompts)]
    return dict(exact_pairs=sum(x and y for x, y in zip(correct[::2], correct[1::2])),
                n=pair_count, exact_answers=sum(correct), wrap_exact_pairs=sum(
                    correct[2*i] and correct[2*i+1] for i, (a, b) in enumerate(pairs) if a+b >= 97),
                carry_exact_pairs=sum(correct[2*i] and correct[2*i+1] for i, (a, b) in enumerate(pairs)
                                      if a % 10 + b % 10 >= 10),
                carry_pairs=sum(a % 10 + b % 10 >= 10 for a, b in pairs),
                updates=dose, target_bytes=dose * m.BATCH * len(prompts[0]),
                params=sum(p.numel() for p in net.parameters()), last_loss=loss[-1],
                held_sha256=panel_hash(prompts), outputs=out, held=pairs)


def attention(arm, seed, profile):
    import torch as t
    from torch import nn

    dose, count = {'smoke': (8, 8), 'pilot': (512, 32), 'screen': (12288, 64)}[profile]
    t.set_num_threads(1)
    t.manual_seed(seed)
    rng = np.random.default_rng(seed + 37)
    # Keys A and B, query A or B, and two distinct printable values.
    groups = np.array([(a, b) for a in range(97, 123) for b in range(a + 1, 123)])
    rng.shuffle(groups)
    held_groups, train_groups = groups[:64], groups[64:]
    assert not set(map(tuple, held_groups)) & set(map(tuple, train_groups))

    def render(groups, query, flip):
        a, b = groups.T
        x = np.column_stack((65 + query, np.where(flip, 66, 65), np.where(flip, b, a),
                            np.where(flip, 65, 66), np.where(flip, a, b)))
        return t.from_numpy(x), t.from_numpy(np.where(query, b, a))

    class Reader(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = nn.Embedding(257, 32)
            self.features = nn.Linear(64, 64)
            self.query = nn.Linear(32, 64)
            self.gate = nn.Linear(128, 64)
            self.output = nn.Sequential(nn.Linear(160, 128), nn.Tanh(), nn.Linear(128, 257))

        def forward(self, inputs, previous):
            context = self.query(self.embed(inputs[:, 0]))
            features = self.features(self.embed(inputs[:, 1:]).reshape(-1, 2, 64)).tanh()
            if arm != 'register':
                # Give pooling and attention an item projection of the same size as the write gate.
                features = features + self.gate(t.cat((features, features.roll(1, -1)), -1)).tanh()
            if arm == 'attention':
                weight = (features * context[:, None]).sum(-1).div(8).softmax(1)
                state = (features * weight[:, :, None]).sum(1)
            elif arm == 'register':
                state = t.zeros_like(context)
                for item in features.unbind(1):
                    gain = self.gate(t.cat((context, item), 1)).sigmoid()
                    state = t.lerp(state, item, gain)
            else:
                state = features.mean(1)
            return self.output(t.cat((context, state, self.embed(previous)), 1))

    net = Reader()
    trainable = list(net.parameters())
    opt = t.optim.AdamW(trainable, lr=.003)
    for _ in range(dose):
        pair = train_groups[rng.integers(len(train_groups), size=64)]
        pair = np.where(rng.integers(2, size=(64, 1)), pair, pair[:, ::-1])
        inputs, target = render(pair, rng.integers(2, size=64), rng.integers(2, size=64))
        start = t.full_like(target, 61)
        logits = net(inputs.repeat(2, 1), t.cat((start, target)))
        loss = nn.functional.cross_entropy(logits, t.cat((target, t.full_like(target, 256))))
        opt.zero_grad()
        loss.backward()
        opt.step()
    pairs = np.repeat(held_groups[:count], 2, axis=0)
    query = np.tile([0, 1], count)
    inputs, target = render(pairs, query, np.repeat(rng.integers(2, size=count), 2))
    with t.no_grad():
        answer = net(inputs, t.full_like(target, 61)).argmax(1)
        eos = net(inputs, answer).argmax(1)
    exact = (answer == target) & (eos == 256)
    return dict(exact_pairs=int(exact.reshape(-1, 2).all(1).sum()), n=count,
                exact_answers=int(exact.sum()), updates=dose, target_bytes=dose * 64 * 2,
                stored_params=sum(p.numel() for p in net.parameters()), mutable_params=sum(p.numel() for p in trainable),
                held_sha256=panel_hash(inputs.tolist()), held=inputs.tolist(),
                outputs=t.stack((answer, eos), 1).tolist(), last_loss=float(loss.detach()),
                scope='supervised reference block')


def central_complex(arm, seed, profile):
    sys.path.insert(0, str(ROOT / 'cx'))
    from qualify_dynamics import Compass, FILES, Interface, Steering, bump, wrap

    root = ROOT / 'cx'
    with np.load(root / FILES[0], allow_pickle=False) as source:
        graph = dict(source)
    model = Interface(graph, *(json.loads((root / name).read_text()) for name in FILES[1:]))
    prior = json.loads((root / 'DYNAMICS_QUALIFICATION.json').read_text())
    graph_arm = 'rewired' if arm == 'rewired' else 'native'
    compass = Compass(model, 'observed_contacts', graph_arm)
    controller = Steering(model, graph_arm)
    turn = prior['compass_separate_timescales']['observed_contacts'][graph_arm]
    steering = prior['steering'][graph_arm]['params']
    count, steps = {'smoke': (8, 16), 'pilot': (32, 80), 'screen': (64, 80)}[profile]
    rng = np.random.default_rng(seed + 31)
    starts = rng.uniform(-np.pi, np.pi, count)
    offsets = rng.uniform(.6, 2.1, count) * np.resize([-1, 1], count)
    noise = rng.normal(0, np.deg2rad(2), size=(steps, count))
    goals = wrap(starts + offsets)
    e = bump(model.phase, starts)
    position = starts.copy()
    traces = []
    for step in range(steps):
        x, y = (e @ model.observer.T).T
        heading = np.arctan2(y, x)
        yaw = np.clip(.14 * controller.command(heading, goals, steering), -.054, .054)
        position += yaw
        e = compass.step(e, yaw, turn['params'], 'opponent', turn['rest_rate'])
        if (step + 1) % 8 == 0:
            # A stationary beacon at zero phase yields an egocentric observation.
            observed_bearing = wrap(-position + noise[step])
            if arm == 'displaced':
                observed_bearing += np.pi / 2
            e = .75 * e + .25 * bump(model.phase, -observed_bearing)
        traces.append(position.tolist())
    error = np.rad2deg(abs(wrap(goals - position)))
    x, y = (e @ model.observer.T).T
    compass_error = np.rad2deg(abs(wrap(np.arctan2(y, x) - position)))
    return dict(n=count, within_20_deg=int((error <= 20).sum()), median_error_deg=float(np.median(error)),
                compass_error_deg=float(np.median(compass_error)), steps=steps, target_bytes=0, updates=0,
                held_sha256=panel_hash([starts.tolist(), goals.tolist(), noise.tolist()]),
                goals=goals.tolist(), starts=starts.tolist(), trace=traces,
                scope='fixed v630 weights and prior calibration, engineered landmark cue, supplied angular goal')


In [ ]:
%%writefile additional.py
"""Eight compact MiniFly development blocks and their matched controls."""
from itertools import product
from pathlib import Path

import numpy as np

from blocks import panel_hash

ROOT = Path(__file__).resolve().parent


def dose(profile, smoke, pilot, screen):
    return dict(smoke=smoke, pilot=pilot, screen=screen)[profile]


def logistic(value):
    return 1 / (1 + np.exp(-np.clip(value, -40, 40)))


def cerebellar(arm, seed, profile):
    """Plastic lag features with local observed-event errors and a supervised readout."""
    rng = np.random.default_rng(seed)
    init = np.random.default_rng(seed + 71)
    updates = dose(profile, 8, 512, 12288)
    count = dose(profile, 8, 32, 64)
    # Each item is a two-cue history. In the held set a cue appears at a new lag.
    groups = np.array(list(product(range(32), range(8))))
    rng.shuffle(groups)
    held, train = groups[:64], groups[64:]
    channels = 34
    same = init.normal(0, .08, (8 * channels, 128))
    masks = np.zeros((8, 128), dtype=bool)
    for i in range(128):
        masks[i % 8, i] = True
        masks[(i + 1) % 8, i] = True
    if arm == 'shuffled_lags':
        masks[np.arange(8)] = masks[np.array([0, 4, 2, 6, 1, 5, 3, 7])]
    if arm == 'ordinary':
        masks[:] = False
        for i in range(128):
            masks[i % 8, i] = True
            masks[(i + 4) % 8, i] = True
    mask = np.repeat(masks, channels, axis=0)
    w = same * mask
    read = init.normal(0, .02, (128, 8))

    def make(pairs, other):
        cue, lag = pairs.T
        x = np.eye(channels, dtype=np.float32)[other]
        rows = np.arange(len(x))
        x[rows, lag] = np.eye(channels)[cue]
        x[rows, lag, 32] = 1
        next_lag = (lag + 1) % 8
        x[rows, next_lag] = np.eye(channels)[(cue + 1) % 32]
        x[rows, next_lag, 33] = 1
        return x.reshape(len(x), -1), (cue + lag) % 8

    held_x, target = make(held[:count], rng.integers(32, size=(count, 8)))
    for _ in range(updates):
        x, y = make(train[rng.integers(len(train), size=64)], rng.integers(32, size=(64, 8)))
        h = np.tanh(x @ w)
        # The feature teacher is the ordinary observation at its preferred lag.
        observed = x.reshape(-1, 8, channels)[:, np.arange(128) % 8, np.arange(128) % channels]
        w += .015 * (x.T @ ((observed - h) * (1 - h*h))) / len(x) * mask
        z = h @ read
        p = np.exp(z-z.max(1, keepdims=True))
        p /= p.sum(1, keepdims=True)
        p[np.arange(len(y)), y] -= 1
        read -= .06 * h.T @ p / len(x)
    pred = (np.tanh(held_x @ w) @ read).argmax(1)
    return dict(exact_pairs=int((pred == target).sum()), n=count, updates=updates,
                target_bytes=updates * 64, params=int(mask.sum()+read.size),
                outputs=pred.tolist(), target=target.tolist(),
                held_sha256=panel_hash([held_x.tolist(), target.tolist()]),
                scope='plastic temporal expansion with local observation targets')


def thalamocortical(arm, seed, profile):
    """Order-sensitive byte composition with matched recurrent and feedforward passes."""
    import torch as t
    from torch import nn

    t.set_num_threads(1)
    t.manual_seed(seed)
    rng = np.random.default_rng(seed)
    updates = dose(profile, 8, 512, 12288)
    count = dose(profile, 8, 32, 64)
    ops = np.stack([rng.permutation(32) for _ in range(5)])
    # Every individual operator is present in teaching. Remove both orders of the held templates.
    templates = list(product(range(5), repeat=3))
    keys = [(0, 1, 2), (0, 3, 4), (1, 2, 4), (1, 3, 0)]
    banned = set(keys) | {k[::-1] for k in keys}
    train = np.asarray([(a, *p) for a in range(32)
                        for p in templates + [(5, 5, op) for op in range(5)] if p not in banned])
    ops = np.vstack((ops, np.arange(32)))

    def target(rows):
        answer = rows[:, 0]
        for col in range(1, 4):
            answer = ops[rows[:, col], answer]
        return answer

    choices = []
    for key in keys:
        for state in range(32):
            a, b = (state, *key), (state, *key[::-1])
            if len(set(target(np.asarray([a, b])))) == 2:
                choices.append((a, b))
    rng.shuffle(choices)
    held = np.asarray(choices[:count]).reshape(-1, 4)
    assert len(held) == 2*count and not set(map(tuple, held)) & set(map(tuple, train))

    class Loop(nn.Module):
        def __init__(self):
            super().__init__()
            self.embed = nn.Embedding(257, 16)
            self.input = nn.Linear(16, 96)
            self.to_relay = nn.Linear(96, 96)
            self.from_relay = nn.Linear(96, 96)
            self.out = nn.Sequential(nn.Linear(112, 128), nn.Tanh(), nn.Linear(128, 257))

        def forward(self, rows, previous, delay=None):
            tokens = rows + t.tensor([65, 97, 97, 97])
            if delay is not None:
                tokens = t.cat((tokens[:, :1], delay, tokens[:, 1:]), 1)
            inputs = self.input(self.embed(tokens)).tanh()
            state = t.zeros_like(inputs[:, 0])
            for inp in inputs.unbind(1):
                state = state + inp
                for _ in range(1 if arm == 'one_pass' else 3):
                    feedback = self.to_relay(state)
                    relay = (feedback + inp if arm != 'feedforward' else feedback).tanh()
                    state = self.from_relay(relay).tanh()
            return self.out(t.cat((state, self.embed(previous)), 1))

    net = Loop()
    opt = t.optim.AdamW(net.parameters(), lr=.002)
    for _ in range(updates):
        rows = train[rng.integers(len(train), size=64)]
        x, y = t.tensor(rows), t.tensor(target(rows)+65)
        start = t.full_like(y, 61)
        logits = net(x.repeat(2, 1), t.cat((start, y)))
        loss = nn.functional.cross_entropy(logits, t.cat((y, t.full_like(y, 256))))
        opt.zero_grad()
        loss.backward()
        opt.step()
    with t.no_grad():
        x, y = t.tensor(held), t.tensor(target(held)+65)
        answer = net(x, t.full_like(y, 61)).argmax(1)
        eos = net(x, answer).argmax(1)
        exact = (answer == y) & (eos == 256)
        delay = t.tensor(np.repeat(rng.integers(113, 118, (count, 6)), 2, axis=0))
        delayed_answer = net(x, t.full_like(y, 61), delay).argmax(1)
        delayed_eos = net(x, delayed_answer, delay).argmax(1)
        delayed_exact = (delayed_answer == y) & (delayed_eos == 256)
    return dict(exact_pairs=int(exact.reshape(-1, 2).all(1).sum()), n=count, updates=updates,
                target_bytes=updates*64*2, params=sum(p.numel() for p in net.parameters()),
                passes=1 if arm == 'one_pass' else 3, outputs=t.stack((answer, eos), 1).tolist(),
                delayed_exact_pairs=int(delayed_exact.reshape(-1, 2).all(1).sum()),
                delayed_outputs=t.stack((delayed_answer, delayed_eos), 1).tolist(),
                target=y.tolist(), held_sha256=panel_hash([held.tolist(), ops.tolist(), delay.tolist()]),
                scope='supervised compact two-population composition probe')


def reward_panels(seed, profile):
    rng = np.random.default_rng(seed)
    count = dose(profile, 8, 32, 64)
    episodes = dose(profile, 32, 512, 4096)
    mapping = rng.integers(2)
    # Each context is followed by thirteen training or twenty held distractors.
    context = rng.integers(2, size=episodes)
    train = rng.integers(2, 4, (episodes, 14))
    train[:, 0] = context
    held = np.repeat(rng.integers(2, 4, (count, 21)), 2, axis=0)
    held[:, 0] = np.tile([0, 1], count)
    return int(mapping), train, held


def working_memory_gate(arm, seed, profile):
    """Rewarded write policy with a frozen state representation and a yoked trace."""
    mapping, train, held = reward_panels(seed, profile)
    rng = np.random.default_rng(seed + 106)
    # A and C encode one sign, B and D the other. Relevance comes from the observed token.
    token_value = np.array([-1, 1, -1, 1])

    def history(rows, mode, random, tape=None):
        gate = np.zeros(4)
        readout = np.zeros(2)
        baseline = .5
        gates, rewards = [], []
        for j, row in enumerate(rows):
            ps = logistic(gate[row])
            draws = random.random(len(row))
            if mode == 'ungated':
                writes = np.ones(len(row), dtype=bool)
            elif mode == 'yoked':
                writes = tape[j]
            else:
                writes = draws < ps
            gates.append(writes)
            last = np.flatnonzero(writes)
            state = token_value[row[last[-1]]] if len(last) else 0
            feature = np.array([1, state])
            action_p = logistic(readout @ feature)
            action = int(random.random() < action_p)
            reward = float(action == (row[0] ^ mapping))
            delta = reward-baseline
            baseline += .025*delta
            readout += .12*delta*(action-action_p)*feature
            if mode == 'learned':
                np.add.at(gate, row, .07*delta*(writes-ps))
            rewards.append(reward)
        return gate, readout, np.asarray(gates), rewards

    independent = np.random.default_rng(seed + 201)
    tape, shadow_gate = None, None
    if arm == 'yoked':
        shadow = train.copy()
        shadow[:, 0] = independent.integers(2, size=len(shadow))
        shadow_gate, _, tape, _ = history(shadow, 'learned', independent)
    gate, readout, writes, rewards = history(train, arm, rng, tape)
    predicted, targets = [], []
    yoke_masks = None
    if shadow_gate is not None:
        shadow_held = held[::2].copy()
        shadow_held[:, 0] = independent.integers(2, size=len(shadow_held))
        yoke_masks = np.repeat(independent.random(shadow_held.shape) < logistic(shadow_gate[shadow_held]), 2, axis=0)
    for i, row in enumerate(held):
        if arm == 'ungated':
            indices = np.arange(len(row))
        elif arm == 'yoked':
            indices = np.flatnonzero(yoke_masks[i])
        else:
            indices = np.flatnonzero(logistic(gate[row]) >= .5)
        state = token_value[row[indices[-1]]] if len(indices) else 0
        predicted.append(int(readout @ np.array([1, state]) >= 0))
        targets.append(int(row[0] ^ mapping))
    ok = np.asarray(predicted) == targets
    reset_action = int(readout[0] >= 0)
    return dict(exact_pairs=int(ok.reshape(-1, 2).all(1).sum()), n=len(held)//2, updates=len(train),
                reward_bits=len(train), auxiliary_yoke_episodes=len(train) if tape is not None else 0,
                writes=int(writes.sum()), reward_mean=float(np.mean(rewards)),
                reset_exact_pairs=int((np.asarray(targets)==reset_action).reshape(-1, 2).all(1).sum()),
                outputs=predicted, target=targets, held_sha256=panel_hash([mapping, held.tolist()]),
                scope='compact scalar-outcome register gate')


def actor_critic(arm, seed, profile):
    mapping, train, held = reward_panels(seed, profile)
    rng = np.random.default_rng(seed + 63)
    actor = np.zeros(4)
    critic = np.zeros(4)
    rewards, nonzero = [], 0
    for row in train:
        # A causal choice on the cue, then distractor actions. Only the earlier action is scored.
        e = np.zeros(4)
        trace = 0.
        initial = 0
        for j, token in enumerate(row):
            p = logistic(actor[token])
            action = int(rng.random() < p)
            e *= .93
            if j == 0:
                initial = action
                e[token] += action-p
            trace = action-p
        if arm == 'immediate':
            e[:] = 0
            e[row[-1]] = trace
        elif arm == 'shifted':
            # Keep the magnitude and move it to an irrelevant observation.
            e[row[-1]] = e[row[0]]
            e[row[0]] = 0
        reward = float(initial == (int(row[0]) ^ mapping))
        err = reward-critic[row[0]]
        critic[row[0]] += .06*err
        actor += .25*err*e
        nonzero += np.count_nonzero(err*e)
        rewards.append(reward)
    target = held[:, 0] ^ mapping
    actions = (actor[held[:, 0]] >= 0).astype(int)
    return dict(exact_pairs=int((actions==target).reshape(-1, 2).all(1).sum()), n=len(held)//2,
                updates=len(train), reward_bits=len(train), writes=int(nonzero),
                reward_mean=float(np.mean(rewards)), value=critic.tolist(), outputs=actions.tolist(),
                target=target.tolist(), held_sha256=panel_hash([mapping, held.tolist()]),
                scope='compact actor–critic delayed scalar-outcome probe')


def continual_fixture(seed, profile):
    rng = np.random.default_rng(seed)
    n = dose(profile, 8, 32, 64)
    feature = rng.normal(size=(3*n, 64))
    feature /= np.linalg.norm(feature, axis=1, keepdims=True)
    target = rng.integers(2, size=3*n)
    a, b, edit = np.arange(2*n), np.arange(2*n, 3*n), np.arange(n, 2*n)
    counts = dose(profile, (32, 16, 8), (256, 128, 64), (1024, 256, 128))
    streams = [np.tile(group, (count+len(group)-1)//len(group))[:count].copy()
               for group, count in zip((a, b, edit), counts)]
    for stream in streams:
        rng.shuffle(stream)
    return n, feature, target, streams


def retention_result(n, target, before, after, final, extra):
    prior = before[:n] == target[:n]
    old = after[:n] == target[:n]
    return dict(exact_pairs=int(old.sum()), n=n, a_acquired=int((before[:2*n]==target[:2*n]).sum()),
                retained_previously_correct=int((prior & old).sum()), previously_correct=int(prior.sum()),
                old_after_correction=int((final[:n]==target[:n]).sum()),
                b_after_correction=int((final[2*n:]==target[2*n:]).sum()),
                corrected=int((final[n:2*n]!=target[n:2*n]).sum()),
                target=target.tolist(), old_pre=before.tolist(), after_b=after.tolist(),
                after_correction=final.tolist(), no_write_old=int(prior.sum()), **extra)


def retention(arm, seed, profile, mechanism):
    n, x, y, (a, b, edit) = continual_fixture(seed, profile)
    w = np.zeros(x.shape[1])
    fast, slow = w[:32], w[32:]
    dual = mechanism == 'fast_slow' and arm != 'single'
    writes = 0

    def read():
        return (x @ w >= 0).astype(int)

    def write(i, target, replay=False):
        nonlocal writes
        if dual:
            xf, xs = x[i, :32], x[i, 32:]
            if replay:
                teaching = logistic(xf @ fast) if arm == 'coupled' else target
                slow[:] += .35*(teaching-logistic(xs@slow))*xs
            else:
                fast[:] += 2*(target-logistic(xf@fast))*xf
                slow[:] += .035*(target-logistic(xs@slow))*xs
        else:
            w[:] += (2 if not replay else .35)*(target-logistic(x[i]@w))*x[i]
        writes += 1

    for i in a:
        write(i, y[i])
    before = read()
    # Same chronological capped observations, never accessed for inference.
    replay = a[-min(128, len(a)):].copy()
    buckets = np.array_split(replay, len(b))
    for j, i in enumerate(b):
        write(i, y[i])
        if mechanism == 'fast_slow' or arm == 'interleaved':
            for old in buckets[j]:
                write(old, y[old], replay=True)
    if mechanism == 'sleep_replay' and arm == 'blocked':
        for old in replay:
            write(old, y[old], replay=True)
    if mechanism == 'sleep_replay':
        w *= .99
    after = read()
    for i in edit:
        write(i, 1-y[i])
    final = read()
    return retention_result(n, y, before, after, final,
                            dict(updates=writes, target_bytes=len(a)+len(b)+len(edit),
                                 replay_updates=writes-len(a)-len(b)-len(edit), params=len(w),
                                 held_sha256=panel_hash([x.tolist(), y.tolist()]), scope=mechanism))


def mushroom_body(arm, seed, profile):
    """Use observed PN/KC, KC/MBON, DAN/KC and MBON/DAN contacts."""
    from sys import path
    path.insert(0, str(ROOT / 'cx'))
    from qualify import Local
    with np.load(ROOT / 'cx' / 'EXTRACT.npz', allow_pickle=False) as data:
        g = dict(data)
    local = Local(g)
    rng = np.random.default_rng(seed + 64)
    n, _, targets, (a, b, edit) = continual_fixture(seed, profile)
    nodes = g['roles']
    u, v = g['u'], g['v']
    dan = np.flatnonzero(nodes == 'DAN')
    mb = local.mbons
    relay = g['usable'] & (nodes[u]=='MBON') & (nodes[v]=='DAN')
    assert relay.any()
    loop = np.zeros((len(dan), len(mb)))
    np.add.at(loop, (np.searchsorted(dan, v[relay]), np.searchsorted(mb, u[relay])),
              g['sign'][relay]*g['w'][relay])
    loop /= np.abs(loop).sum(1, keepdims=True).clip(1)
    da_kc = (nodes[u]=='DAN') & (nodes[v]=='KC')
    address = np.zeros((len(nodes), len(dan)))
    address[v[da_kc], np.searchsorted(dan, u[da_kc])] = 1
    address /= address.sum(1, keepdims=True).clip(1)
    if arm == 'permuted':
        address = address[:, rng.permutation(len(dan))]
    edge_address = address[local.u]
    odor = np.asarray([local.observe(seed*7+i)[0][local.u] for i in range(3*n)])
    groups = np.arange(len(mb)) % 2
    idx = np.searchsorted(mb, local.v)
    w = local.base.copy()
    rewards, writes = [], 0

    def response(i):
        y = np.bincount(idx, weights=w*odor[i], minlength=len(mb))/local.post_norm[mb]
        score = np.bincount(groups, weights=y)
        return y, logistic(30*(score[1]-score[0]))

    def read():
        return np.asarray([int(response(i)[1]>=.5) for i in range(3*n)])

    def train(stream, edited=False):
        nonlocal writes
        for i in stream:
            y, prob = response(i)
            action = int(rng.random() < prob)
            target = int(targets[i]) ^ edited
            reward = float(action == target)
            rewards.append(reward)
            gain = 1 if arm == 'open' else (1+.5*np.tanh(4*loop@y))
            mod = edge_address @ np.broadcast_to(gain, (len(dan),))
            elig = odor[i] * (2*groups[idx]-1) * (action-prob)
            delta = .75*(reward-.5)*elig*mod
            w[:] = np.clip(w+delta, .5*local.base, 1.5*local.base)
            writes += int(np.count_nonzero(delta))

    train(a)
    before = read()
    train(b)
    after = read()
    train(edit, True)
    return retention_result(n, targets, before, after, read(),
                            dict(reward_bits=len(rewards), reward_mean=float(np.mean(rewards)),
                                 updates=len(rewards), writes=writes, params=len(w), observed_loop_edges=int(relay.sum()),
                                 held_sha256=panel_hash([seed, targets.tolist()]),
                                 scope='v630 contact-based MBON–DAN gain analogue with scalar outcomes'))


def grid_head_direction(arm, seed, profile):
    """Fixed toroidal transport with or without attractor sharpening."""
    rng = np.random.default_rng(seed)
    n = dose(profile, 8, 32, 64)
    steps = dose(profile, 32, 80, 160)
    # Even the maximum allowed path fits within the unambiguous half-period.
    bins, span = 16, 8.
    angles = 2*np.pi*np.arange(bins)/bins
    kernel = np.exp(5*(np.cos(angles)-1))
    kernel /= kernel.sum()
    xgrid = np.exp(1j*angles)
    true_yaw = rng.uniform(-.12, .12, (steps,n))
    true_distance = rng.uniform(.012,.022,(steps,n))
    pairs = (n-max(1,n//8))//2
    true_yaw[:, pairs:2*pairs] = true_yaw[::-1, :pairs]
    true_distance[:, pairs:2*pairs] = true_distance[::-1, :pairs]
    true_yaw[:, -max(1,n//8):] = 0
    true_distance[:, -max(1,n//8):] = 0
    observed_yaw = true_yaw+rng.normal(0,.004,true_yaw.shape)
    observed_distance = true_distance*(1+rng.normal(0,.04,true_distance.shape))
    state = np.tile(kernel,(n,1))
    grid = np.tile(np.outer(kernel,kernel),(n,1,1))
    true_heading = np.zeros(n)
    position = np.zeros((n,2))
    indices = np.arange(bins)
    path_len = true_distance.sum(0)

    def shift(values, offset):
        low = np.floor(offset).astype(int)
        f = (offset-low)[:,None]
        a = (indices-low[:,None]) % bins
        b = (a-1)%bins
        row = np.arange(len(values))[:, None]
        return (1-f)*values[row,a]+f*values[row,b]

    def diffuse(values):
        return .7*values+.15*(np.roll(values,1,axis=1)+np.roll(values,-1,axis=1))

    def sharpen(values):
        shape = diffuse(values)
        if arm == 'attractor':
            shape = shape**1.10
        return shape / shape.sum(axis=1,keepdims=True).clip(1e-12)

    for yaw, distance, oy, od in zip(true_yaw,true_distance,observed_yaw,observed_distance):
        true_heading += yaw
        position += distance[:,None]*np.stack((np.cos(true_heading),np.sin(true_heading)),1)
        if arm == 'reset':
            state[:] = kernel
            grid[:] = np.outer(kernel,kernel)
        state = sharpen(shift(state, oy*bins/(2*np.pi)))
        heading = np.angle(state@xgrid)
        # Operate on the two-dimensional population state, not on a position accumulator.
        for axis, amount in [(1, od*np.cos(heading)), (2, od*np.sin(heading))]:
            moved = np.moveaxis(grid, axis, -1)
            plane = moved.reshape(n*bins,bins)
            plane = shift(plane, np.repeat(amount*bins/span,bins))
            grid = np.moveaxis(plane.reshape(moved.shape),-1,axis)
        if arm == 'attractor':
            grid = grid**1.045
        grid /= grid.sum(axis=(1,2),keepdims=True).clip(1e-12)
    decoded = np.stack((np.angle(grid.sum(axis=2)@xgrid),np.angle(grid.sum(axis=1)@xgrid)),1)*span/(2*np.pi)
    diff = decoded-position
    error = np.linalg.norm(diff,axis=1)
    moving = path_len >= .25
    return dict(exact_pairs=int(np.sum(moving & (error <= .2*path_len))), n=int(moving.sum()),
                median_normalized_error=float(np.median(error[moving]/path_len[moving])),
                order_pairs=pairs, median_order_separation=float(np.median(
                    np.linalg.norm(decoded[:pairs]-decoded[pairs:2*pairs], axis=1))),
                zero_motion_drift=float(np.median(error[path_len==0])), updates=0, steps=steps,
                target_bytes=0, state_size=bins*bins+bins, outputs=decoded.tolist(), target=position.tolist(),
                held_sha256=panel_hash([true_yaw.tolist(),true_distance.tolist(),
                                        observed_yaw.tolist(),observed_distance.tolist()]),
                scope='engineered toroidal transport, attractor sharpening versus diffuse and reset controls')


In [ ]:
%%writefile sweep.py
"""Run the MiniFly development sweep with durable per-job receipts."""
import argparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from hashlib import sha256
from importlib.metadata import version
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import tempfile
import time
import traceback

ROOT = Path(__file__).resolve().parent
SEEDS = (284813, 516197, 873411)
BLOCKS = {
    'cortical': ('sign', 'dfa', 'bp'),
    'attention': ('attention', 'register', 'uniform'),
    'central_complex': ('native', 'rewired', 'displaced'),
    'cerebellar': ('grouped_lags', 'ordinary', 'shuffled_lags'),
    'thalamocortical': ('loop', 'feedforward', 'one_pass'),
    'working_memory_gate': ('learned', 'ungated', 'yoked'),
    'actor_critic': ('trace', 'immediate', 'shifted'),
    'mushroom_body': ('closed', 'open', 'permuted'),
    'fast_slow': ('coupled', 'single', 'uncoupled'),
    'sleep_replay': ('blocked', 'interleaved', 'pause'),
    'grid_head_direction': ('attractor', 'diffuse', 'reset'),
}
PROFILES = ('smoke', 'pilot', 'screen')


def digest(path):
    return sha256(path.read_bytes()).hexdigest()


def verify_bundle():
    manifest = ROOT / 'SHA256SUMS'
    for row in manifest.read_text().splitlines():
        expected, name = row.split('  ', 1)
        if digest(ROOT / name) != expected:
            raise ValueError(f'Bundle integrity check failed for {name}')
    return digest(manifest)


def runtime():
    return dict(python=platform.python_version(), numpy=version('numpy'),
                torch=version('torch'), platform=platform.system(),
                machine=platform.machine())


def jobs(profile, blocks, bundle):
    seeds = SEEDS[:1] if profile == 'smoke' else SEEDS
    env = runtime()
    for block in blocks:
        for seed in seeds:
            for arm in BLOCKS[block]:
                spec = dict(block=block, arm=arm, seed=seed, profile=profile, bundle=bundle, runtime=env)
                key = sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]
                yield f'{block}-{arm}-{seed}-{key}', spec


def write_atomic(path, record):
    with tempfile.NamedTemporaryFile('w', dir=path.parent, delete=False, suffix='.tmp') as stream:
        scratch = Path(stream.name)
        try:
            json.dump(record, stream, indent=2, allow_nan=False)
            stream.write('\n')
            stream.flush()
            os.fsync(stream.fileno())
            stream.close()
            os.replace(scratch, path)
        finally:
            scratch.unlink(missing_ok=True)


def worker(job, spec, output):
    from additional import (actor_critic, cerebellar, grid_head_direction, mushroom_body,
                            retention, thalamocortical, working_memory_gate)
    from blocks import attention, central_complex, cortical

    runners = dict(attention=attention, central_complex=central_complex, cortical=cortical,
                   actor_critic=actor_critic, cerebellar=cerebellar, grid_head_direction=grid_head_direction,
                   mushroom_body=mushroom_body, thalamocortical=thalamocortical,
                   working_memory_gate=working_memory_gate)
    cpu, wall = time.process_time(), time.monotonic()
    path = Path(output) / f'{job}.json'
    try:
        args = spec['arm'], spec['seed'], spec['profile']
        if spec['block'] in ('fast_slow', 'sleep_replay'):
            result = retention(*args, mechanism=spec['block'])
        else:
            result = runners[spec['block']](*args)
        receipt = dict(spec=spec, status='complete', result=result)
    except Exception:
        receipt = dict(spec=spec, status='failed', error=traceback.format_exc())
    receipt.update(cpu_seconds=time.process_time()-cpu, wall_seconds=time.monotonic()-wall)
    write_atomic(path, receipt)
    return job, receipt['status']


def read_receipt(path, spec):
    try:
        record = json.loads(path.read_text())
    except FileNotFoundError:
        return None
    if record.get('spec') != spec:
        raise ValueError(f'Unexpected receipt identity {path}')
    return record


def launch(job, spec, output):
    command = ('import json,sys; from sweep import worker; '
               'print(*worker(sys.argv[1], json.loads(sys.argv[2]), sys.argv[3]))')
    subprocess.run([sys.executable, '-B', '-c', command, job, json.dumps(spec), str(output)],
                   cwd=ROOT, check=True)


def report(roster, output):
    counts = {'complete': 0, 'failed': 0, 'pending': 0}
    panels = {}
    for job, spec in roster:
        r = read_receipt(output / f'{job}.json', spec)
        if r is None:
            counts['pending'] += 1
            continue
        counts[r['status']] += 1
        if r['status'] != 'complete':
            print(job, r['status'], r['error'].splitlines()[-1])
            continue
        result = r['result']
        key = spec['block'], spec['seed']
        signature = result['held_sha256']
        if panels.setdefault(key, signature) != signature:
            raise ValueError(f'Paired arms used different test inputs for {key}')
        score = result.get('exact_pairs', result.get('within_20_deg'))
        print(f"{spec['block']:16} {spec['arm']:10} seed={spec['seed']} "
              f"{score:2}/{result['n']}  CPU {r['cpu_seconds']:.1f}s")
    print('Roster status', counts, flush=True)
    return counts


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=('plan', 'run', 'report'))
    parser.add_argument('--profile', choices=PROFILES, default='smoke')
    parser.add_argument('--blocks', nargs='+', choices=tuple(BLOCKS))
    parser.add_argument('--workers', type=int, default=2)
    parser.add_argument('--out', type=Path, default=Path('receipts'))
    args = parser.parse_args()
    if args.workers < 1:
        parser.error('workers must be positive')
    bundle = verify_bundle()
    selected = args.blocks or list(BLOCKS)
    print('Blocks:', ', '.join(selected), flush=True)
    if args.command == 'plan':
        print(f'{3 * len(selected) * (1 if args.profile == "smoke" else len(SEEDS))} jobs in {args.profile}')
        return
    if len(selected) != len(set(selected)):
        parser.error('List each block once')
    for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
        os.environ[key] = '1'
    os.environ.setdefault("KMP_USE_SHM", "0")
    roster = list(jobs(args.profile, selected, bundle))
    out = args.out.resolve() / args.profile
    if args.command == 'run':
        out.mkdir(parents=True, exist_ok=True)
        pending = [(job, spec) for job, spec in roster
                   if (read_receipt(out / f'{job}.json', spec) or {}).get('status') != 'complete']
        print(f'{len(pending)} to run, {len(roster) - len(pending)} reused', flush=True)
        if pending:
            with ThreadPoolExecutor(max_workers=min(args.workers, len(pending))) as pool:
                futures = [pool.submit(launch, job, spec, out) for job, spec in pending]
                for future in as_completed(futures):
                    future.result()
    counts = report(roster, out)
    if args.command == 'run' and (counts['failed'] or counts['pending']):
        sys.exit(1)


if __name__ == '__main__':
    main()


In [ ]:
import base64, zlib
from hashlib import sha256

_payload = {}


In [ ]:
# Embedded cx/EXTRACT.npz
_payload['cx/EXTRACT.npz'] = (
    '...'
)


In [ ]:
# Embedded cx/COMPARTMENT_MAP.json
_payload['cx/COMPARTMENT_MAP.json'] = (
    '...'
)


In [ ]:
# Embedded cx/RESOLVED_BOUNDARY.json
_payload['cx/RESOLVED_BOUNDARY.json'] = (
    '...'
)


In [ ]:
# Embedded cx/SYNAPSE_COMPARTMENTS.json
_payload['cx/SYNAPSE_COMPARTMENTS.json'] = (
    'eNrNnUtzG7kRgO/7K1w6r0d4P/aYqlSuuae2WCNyKLEikyySiuNN7X9PQxZtGZiGekaN8V68K4oiPwwa/Qbwv18+fLj51F+G065/3P3RX3aH/c1vH5wWv6bfPJ0e4aebh8vleP7t9vZ8OZz6+6G7Px'
    'zuH4f+uDt368On2+3jl8+70/Bx01/62/VhM/z39vl/t/327hY+6vb8Zd8fz8NqfTicNrs9fB384fk/3f0fN89f8+335/+s7v/YHVfnh15Zl756s5Xab4ao4p1aK+/72K/v3PpuE0Jvet/b9VrebYXc'
    '6KEftnKtZXCD3a7dNsrtnbNfv6C/PPbnV5/qrYNPXZt+GKSMSsr+TgkfwqYX6dNtL0yETzZ3m34b+0HE9UZ6+DZj1ltt7uTXTx2O9/Bh/4L//fDhf8//wounw+Gy2m2ev0UJ6200wgkThIxRP//d89'
    'suX45Des/f//mP7y+ed5vnF0+7+4fL95f//jd48foF8PPxoYeHtRnSt3/0vrMu2uiVV0GrYH79/sbTcH56vPT7C7xRdDF45aJ2zhsdjZKv3rjbH58uMD37S7++nOHdUqr48us/n//756+UYdoAU+Qk'
    'cZiPw5Y6SgujNCLC5zshpI+uMkothTTWCxW8Mao6SuH0jFHCc1TaS/7JlNZ2IjoVYTYDCGaw+DCtFMYYoTQsMK1dfZhaTR8mfL7VRnn+yZS2i8Z7L11QAR6llvgwTZAiRBmUdMbJuswqa+YMM8LS1K'
    'LBMI3uVDRW2mB90jIKH2Zak15KF5PU+lgbprczZBbWs3QixgYKSIfOeWm9MiGA4IqIj9JbYaVMakgHJd6QWT9nmPC5MQrRYGk62+k0OcpJK5SylcmUFjSxA7kOsEBtqA5TRzlrmNoZHRrMprSuMxHM'
    'n48hamOjrggtiGyAJRzAvkVXX5tu1nQaFb0zosU4pe6i9sEGr+HblKyMM2qd5jGYKD2Yldo4w6xRRuWMtw00kA+dh4WvtA1ReVexmjqAwTGgjGXQRtYVkA8zBmlBlIKxDVamCB34VmAhTFAWdLlHRw'
    'mCarVN0xlASWhfl9hZRtOBchDO8Q9ThU4rCZOkLfhBYPcrjp4DTQwWBZ4MTLusDtPNG6SLTjRwgJTvRAAPPlotvfQqVhwgBa4muJvagaoCJVEbJPx+xijBkwUPjN+ZBfcHzAdoHqdhIr1U1WXppXEC'
    '/HUDTq95w2LOMSXBgatvVQPlA46B10IYLS2sN1+RV2/AAXNJuyr7o5YqlY+co2FhtTuIj1rYEaU6Z5NTAC67c7VVGcDFE8pZr5UXsu79yDkCC0Gy066FHRHg/PgUJ/sUFcDqrwWZVgTrFHj2GjRtXW'
    'KFnzPM6FywpsVkmtAJkWYTnHAjZSUucUqkOBDWJTwO84Yr62bMJnw+GJLYZJii0+CGa53cNqOE1lW/AFCcklGCUq76shCrzRkmCEoUroXzozqQMAvG0GijYcYqwTRMITiCAhxemNW6xw6e4JxhOnDx'
    'TAOT+dHC2gwCIg0fQQHA+q84shACWh0hWoPYtOrH6hnZHyXh+Qkf+KcyQoiZBDHJKkxozZQEMJgQG5mgn5NRdSUr5sykgo/3LbI/H7XoQJk4LaNxoDqDr1hMMCDJsTfOgRKqW0wzR/uAbnBgtvnn0i'
    'kQV29hlowNpuL6gFcH/rqyQGNBsutrUs6ZSR3ADfOa38EzvgugSJTTKesbYqhpnhSgaRVj8qmrcRcYgxmDNCmhqE2DQYoO4ksF68GmbF4tgE45Z9A7YCA8hJnVRan1nDH6YFRokMKzSQ5hrVn4qgCj'
    'rIkrhFrWm2fto1zdUM7x7hRErhD78U/kR2U6cF4dLMeUUg7gxtbcHliXKjns4CDZutsDwj9jmKD+rBItss7GdRYm0RmIoMFt8xV/INkxC7oYBul+TPWN6J45qgcsVAQF0SQg0V2QEoxfDFoZ0OMVYw'
    'kaB2J5H1JsWR+mmjWXVoM/0qDqZUwHCy0VKGSI4BHIyiAhvoYRwtJMeQtZzxOIOQszafCgG2S2wLOTEBVrn5JWKWNQEVhYmV6CyYTBQixaTVGKOHeQkT9N4EUXTCpNQLiVcgAVK+IdhJ/wNMCbB5Vc'
    'jSzdjLgSnp8R1rZw7LzuYC1EC84peN/WhVr1MgRlINpK+uGtJMGcbLMKJmrbIheiOgOyBQYEQikpQYdX/FcLLh1oJ+mNSf95Q8GqOcMMED+bBtk777qUanPepTXnQ83vAUmFRSkDrB5T93sgrIyzRm'
    'mSH90gqy59p6VzME0mmYlKxR3iMnAFbEw1cSXCG2WgGXOppRchaNfCWjqwJBp0Gww1uVg1T10HB+bMK58MW+B3fbR6Lng1yN9F2YGfbpRXFhSMq6UIkisAcy8EeEAw2Kq1nCOxqUPFCNHC8wmuE2Cn'
    'IO43Ar6mZkoC+LrGWwigQcvqtyq0c4YJayGFO03CLuFSatBqkERZk1gNkg2+gxQiVffqToGcI7AWRCoGfqdAatnFlGqxQguRnBpc+aTSrQQVGFIBsF64jHNcdRhkKl7yB5fOJQULZiqmXggYLD7GkJ'
    'LNGjxZ0A/gC9abYGZpWOd1m2Sz78BG+RR3gXvlRa2gB4opgIfuU4ZPhjdcH2VnjDKE1IDSotYlO1A5AsJia40HxVgLoQ3YEfA0BcxlvdYFa31GPGIghIbAqMGqDF1yYMGnCeHZTtSCLilgOmGUxok3'
    'ypaggqcPMuUqJGC0KFvCXIL/GqOBFWdMLUXpYkoTgGCBzXShqntCbizh39+vXaSr47Bf9ent+2GdWnDPZV/p+fB0Wg9ZJ6IOShv3eqr70/1webOR73DpH5+z/N9eGu5+/PmYfhajE/IOkrIL7YXku4'
    'b/SvLq5zYkZQvDC8n3DMFXklc/tyEp2wyuJNnsyMazM1JxfCGx2TOxjZ/JSI3lKicikxPRmKSsg1xnR2ezoxuTlEnuK4nKSFRjkjIT/UJiMhLTmqRMMF4lNhPYxhxl6uiFI1NrjbXaSG7nqukzEN2Y'
    'ZMShvJJkU6Mbz82I03eVEZ8JiW9LMuKYXbVa9kzU/GdSbjQZIQEPW6cYuKG8kjikUCbCYxnRaTLTabIxSSruyTGSzAjHthxKQhDxYzGB3xugkVgDbrDwTS0fiUSnJk8vRrS8zMRVzpfXckWMrZuS90'
    'qS6RLpG5OUO62uNjibHaPbklSii2wNa9mYBI0udOaXaNWWBPek2fQrkaPoOuP2j2gcuD+fRRbKNCZB/fnFZQT155dewbg/rzOtphtrNdyPZowsSCQjtRhuf4DIUZRLrhYnW8Ky8RrG/WhGX620apO8'
    'VxUybRLaklT8V0Z/gEJS8RuzjIkUjUmSLw2KrSTJHkm2gg0zB+41stngch/4qO3zXkntG9pgGkew1ks/lsnKVo0MbUlAk5i0j+RN3ZrpEc/MYdIj0WOSqnJRnb16RzTEpLhCZXZPqcYkaFyhMynRoS'
    '0JHlfw+YxEEjSu4PPUaCR4XFF4A9kzkdwkC2ToiSS4R8+W8ySSoB49X2xBJEE9ej4PiUiCevQ6W8W68SrGc+NcdpjGgUcWmdHxrTnQyIItq0YkwSMLRpLCmoxZvzKPz+7PE0nQGIdPWikclQiHrepI'
    'I1EqSlg7omUkTCRBI5zFn0kIzgUR3476MiMcJoAUxyONlk+8Sj59S3GlcLQPtogcaPHEVUMcy83hjItxVEAYRZVEEqMFV023bLihkWjvDHiMI88k843mu0ZlwDDJ2CzMgafT2MrSNBLc2PC1hxFJqO'
    'Xg2SkKGgc5mTbfHSlPTJuk3/mKwTQSXMNzJeaJHGg6TWdPRDd+IgvoMxoHmtarJ34tM0clrccX/JYH8E1aNYyySuDA1wxfUwmRBF01tuqvGm4O1Bth1KsEDnztMurV8njBaRqezz8jkVSklS/Qo5Gg'
    '3nOsSmvk5kBXDZu0kjgq0rqwjOA6nqu4N3Je5WjKN+2D8QTf+R2alUaCl/cElxtAAsHdgLzzN69laWYSXEiK4Hd2x8LIPpLRyUkH541lwRlDLCIJljFirNuUO1qm5VrZOhZoJHhmkc1No3G0D/ZoHK'
    'i0MmbBiSSotPLVxcsDUN+T0JvvuNI48IQenytPI0FdaEZNUp5o+p7dDPPtDYkDTxoxukckkko3GF9FjUaCrhvGFUwjwVfO0rODp8L5kvIkksrOCr5Qq1wV0zqgGLUJiQTtgOKUEwrJyBw2WDsUkpGT'
    'XX/SM8H7jtgCPxoHvi+YrRuaSIJ2HbF5rzQOtNNHZakspduS4B02fJ2uVBKsx4axh5FEMnJeBvu6KRKIY9q17MzlT3iSSEaKxQ10GomkLGtcpSTTJUo2J8mT1j+LpEwDNvAHSCRlNMSuX0kc5bUH/D'
    'kkEsnIkf3XDF82N7rx3Iwc8viTpGTkiMIrSSYmqrGcjJw8x10OJnHgFoctm1WuiJ9V2CKRLFKGpZFQs1muMQcakzM2/NBI8JicMRKmkFSKbGwkI/tIpmXo2bJINBK8MSzf4jE3w0fjqPQgs0V9RBK8'
    'B5lxbgo7P02/Mj4TCskSjQNEkuYFexpHRZfwSUmZG5q0grkas4kceI6ezQbTSCo5er51QyPBc/RsVphIgufo2WrCRBLUH+DykGgclQw9Wy5rRBKnZejZIhwiCZqhZ9OuJA58XzDrCqaQoJUCvl2WNB'
    'K8UlA03zaeHTxDz5cbIJLgJ3cuLCeVHD2npieQ4Dl6vmwWlQTN0bO1i9NI2mdMRqpn0+I+PnmlkeBbk9jy0TSSytakpUlQDzZUO5EsM8cisTCNBI2F2Xw1Egfuq3FtbhiJcCdVtthiLRoHWk3iy7wS'
    'SdBqEqesUkjw2glXdp7IgVZOlpURvG7C1wtFI2lfIR/RmNPOPeLz0WgkeHzDODckEvS2hoXnBr+rgS8/QSLBT4Liq8ESSdCOLE4tTyLBe7KWnh30/CXGrA2NBL0hYdmVg5++xNVFQePA7yVgrBHQSN'
    'BTj/jOzaaR4PcSMD6Tcm3+pKMoaCSVeI+tc49IgsZ7bCuYxNG+Z4DGUcmI82l5HawKXs+tN9b3BDluDnQTKptupXGgVU/ONUMhqRwBwUdSZnUnaTS+vXQ0ElyjMXqMNJL2Go3EscSZ2UQSagZr9nFh'
    'IzpiWnc24xOhkOA90Wy5cBpHudP8OjPZ1FjbmAS/sVAvOzcjOph9/ZI40NimiLJMaxJwBEY1Sb5TS/olSOLbp9zqxhzlxfRXGcmeiG1MgvdD5/uS3tHxQyMpr8vGvBLVnCS/0/p6onnkOtGcRoLHnm'
    'y6pMzFTKp58u1eI5LgHiNfNE4iqZz9y5fFIpEsUmukkeB9t3zZThoJejZG3Wec0BNdRvyT+sIY81gkErwLinPl0EiwvVps1TUSR2XfNl/0SSNZ4B5lIgnajcVncUgceC8WX66TRFLpxeLrlCOStN8v'
    'TSPBq8FsK7i0JNOqwXxdtzSSJarBNJL21WAaB14N5vOPSCR4NZixO41GglaD2fI2NA68AsvnC9BI0Mono54nkeBR37JzU7kbnu3obCIJWoPluwGORlKpwfJFOOXumkndg5wWh0SC9g/y3UVHJAH3yD'
    'sj/gKzg55GwXfbGZEkiuTEqrbWj0aC7n5h1Cc0EvSMDrYTu2gklciPL96ikThjoggNrzmhcSzSaUojwc+nWpwEvQNuYb2Gx1sms8TmHZa4qBdNygRznfRK48C7OjjnhkSCnwfMqOlpJNTzgN9RyyGR'
    'kO8wfscTKWqK75MS0ZgElxJTbUOS7CDopXiBL/SjkOBCkrdTzNdpZRZxUk3JL8qxwCkQNA78FAg+C0wiWeQUCBoJ2t/J5qfRONCTFwKX/SVx4H2mbGum1JjT+n4MmydPI0Hzncs+kUrfD1+XGo0Eze'
    '0t+0TImT3TmAPNp7HpdxJHJZvGp9/LuZ+2l54vD00iwW2vr/YLOGaOJe5xJpLgvRx8tpdGgvrObLaXyIG6znxZo3KXwKROZJNJibFtSfBOZJPZX2OakyD5cD5rQ+LAd9MzalcSCdqdne9peMfZLSQS'
    'fF8/Y46ERoJ2Z5tsDZvGswNGWFrVdMcYjQM968Bket7I5iRIV7Q1bF3RJJJKNpzR9pFI0P7sosrXWE7w/uxc0+vYnATx5zntcLEqpp3uyqjrKSSV0135Om9pJOgplfUMReDmIF/5+Z44mEJSOWeWUZ'
    'tQSCp3fnI+k1yLT7salm+3B40ElZPA5jdSOCrpcFWtEGhmEPLtwe8ILYpAapL7yjgxFA5nU/+g+wtwYI2dmRqxoTEJmuo0is+JppDgrZ1+0bnBk66cbhqNBNtsWWw4EY1J8NZOvrZbIgkWWPAlkCgc'
    'uONatKfpxiRok2mxxTE2JkGT4svOTWWrZaZK3HwHuizeTWsx5WuJppHgLab5funWJGiLKZuUEDmw5N7CHHij69Izgze68l2iQSRBL4TlO6qaSII1uhZJRt2WZJFGVxJJJaXGd7UIjQRtMGVLvBI50P'
    'ZSvuQRiQRvL9WRK7U3Mv8/6bokGgmeUOPb6EEkwS9fYyue0EjwRBbfhqCR1TmpZS9Uy/WemaN9yx6RA2/ZY2sboJFUWvbYtBqRBL+4iW27CZEETYlzWRwiB6pJlpVWvHmQb0v9yHf8pGubiCTotU1c'
    'x1TSOCpeNNshA1SS1seE0DgqlzYtLCOVY0IYdSuJBD0mhCvvSuRY4MomGknlmJCF1w0eV/AdZUojweMKPp1WjPWneUcUkop3xBaPE0lQ74gxqqCRoF5JrDYMRGaORbySsp1nUhTMZvlIHJUWCj57Qy'
    'LBi+PLPpFKAwVbFmuk0Wra1ka2Vg4iCS4lfDqNRFJpobDLktAbbWZv9xwpJk7rswmMAkshoXbxu9YcaBe/ZVNqJJKKuEau0slIoXdSGiuPcGan00gclbnhU2plEnNaEZateYFIghZhGQMLGgm+s4Gt'
    'x5RIgpY/GR0TGgl+pg1bsZ5Ggu+x4DsPikiC7ihYeu3gRUedyYluLCd4KMxY2CrtyaS9Y1wtDDQOfOfYwhz4HRZ5M2OY7avRSPC97GLRR4JvkuLqeCVyoF2VxbHzsjUJeoUF2zUJRBK0l9FnisSJti'
    'T0qyPmXwpAJMGvjohslT4qCXZ1hOPaTDfSMTlpaz3fwfM0Ejy4YKt+kjjwrfV8/WFEEnRrPVtenMaBhlv163AUmWOkq3ba9SJsmVcaCS4jXHUtIgf54DI/95wfIgh+CgRbco9Igp+gxtZoSiVBsxRs'
    'SeCRBOIkX14brg0FNBLcmy/2bpvmJEi+hCuuIHLg2RK2iI9Igp8Dkev5xnNTOQeCLb9HJMHPgchPPWhNgmZL+M6sJpNg5y9orvMXaCSVZnFGq0MioZ+/4BuToEFOsYplcxIkyOFKD4xkyya1VfI1uh'
    'BJ3tuiDf/+nn5xs71bHR/782W33l2+rNaH/aVfX1brh2H9b3j3V9Sb8+X0tL48nfrH1brfb3ab/jKcn3fbvsz8zbHfnc6rz7vLA3zG4bTZ7V/eAmMZe8vz15xXu/15txlW27sf3vnpsHl67C+H0+rr'
    '3+z2q/7+/jTcw2e+utq1eN/I17/Mw83mcOw/7fbD6X63Hv/66ztPw3p33A37y8vn0f/y26N5jbMf+tPdtwf77ZHC2+UPHSbqh59sLjM3m9350u/Xz4P61+8vL34+7WCYq2Hf3z0OG/jNtn88D7+8/N'
    'XNbr8ZjgP8s7+s7ofDp+Fy+pJP7dMpSc3Nw+VyPP92e/v58+du9+nT4QTStH380vXH4+3msD7frnen9dPucvsicTfnh15Zl/40rMEVc3cbFc0AivROb7Z3bjsMvR7ubC83chvkVkUvwzpE10sZt0b6'
    '3lq1MW7orzr3pn/c3e8/Jdb18Ph4flXVuXkYHjffXr0Gy19f/TRsdv1+tRnunxVRZ1VaGFKnRJ4Q6vVbj1G8vM/4DrwCZaOSCmJBczV0Lx/Z//flfVZ0UfkgtFQKjKYpSJ/Ow2YF03t62oOEvJ6AX/'
    '785f9zXxdv'
)


In [ ]:
# Embedded cx/DYNAMICS_QUALIFICATION.json
_payload['cx/DYNAMICS_QUALIFICATION.json'] = (
    'eNrtW01zG8cRvetXoHhWtqZ7uqdncpNtORdbVklKKq6UC7UAFiRsEkAA0I7t8n/PG4gisfhY7hIQRSfiQSUssDvTb7r7ve6Z/f1Zr3d2vijnF/3JdH69Wp79tfc7ruHqy3++e/Piy3fFdP4bLp5FGz'
    'rjkaSovhoJmdOxqTfjsrRBpRrE/EAHMUg1YlYbV8zVgC0lN4xhePb8/WO//O7b1y/evPv25at3/W9fvC5+XM6m+fkyTgMMMCQ3cmbRVaNBJaOS2Y9VRiYs1TDycMQpqLqSByMe6Cgwl/jp0OuH5795'
    '+fa7b/7x8qv+F9/9/dVXL958fztCDOSFB4Oxr1j8aDhyVUnJ8wAWDKoYoo3d2Akljkl5PB7KWGQ8rlwcOG+j8YcR3n7/6sXrty/7G5a8vR2ksuHA4ekjN6DRULziWcmP3FBiGlYjr5oSUSlCWo4G3m'
    'LEhAYwe8xUhlieYYw/8kBn83JRXlWratGfLUbV4m5hhrOrebnMK/Wv9YW8Vq//1ltUw+vFopoOq5uJ4vpX1eWqtN5kejEZTFYTTPH2q9cvX/XGFVAuhz/dXf21/KV3Xk6mZ+sLP9xYvFxV1WIyPd8c'
    '8usvevOLcln1ZuPxslphjN6iHE3K6XJn+PUDby9+/SVvXXn99Te+t7pYVMuL2eXoZuhbHO7MvbG/PD9fVOflqrq9hIvTcjX5efPKBwQ3ccp/rkj6fOMzFfy89nWofe1d4W4//nD3TUaqfzUb5SHPZv'
    'P5bFpNV2cb3w/Ly8lgUa5B35zVGs715XLxa//GMnLPD/wg30yh9u3qejHtX1UZ6/6oyosSUhGCwIEJfiVKcff3+Tmed69/mMDuN4tqhVWqRrs3bkyvNg2SwjPCXT2HGBJ8//auPzaQ+fc1oBlPhi2x'
    'SUdBI1JER+S9hUAS9AA0KRyEhrkRm607D2JjBXJiEiahIPi7g+bZFkRnd5HcR3D137t2/zYfLOtevovYhjM14bXPkWJhKlg5pKeQYtz+8TZUNaDc9vUNlA5MqTY4FxGDJlLCQkV1tAeYXyaLavSYiC'
    'RfqESk6RDg2+b9o0JCHAvykTmpSyn/s4PJdNbPqagzJOnBkGgqhJwkZvC0RLVOkJA0YFK7q2so7fhJCzZA9m9iAyrc45PBcVyQXMFJzHkIDo211TkJF5C2yncRlJTYg45ScN75k1DBcchgSjF6j+ko'
    'XAcE1ZkKXCMyzG2RiQaS9oR1EvZhlwmebeB0Nh/054vZfyZXk9WvD1I8VKjWJY7UP/rH9/F41EpyKFAFgCuCmXNiD3HyZlZn347VYxHJJx+g6QM5jidxczsOHF8gOWpiI83hx53d3FsjNtZO8WAeHh'
    'kgOBC6hhj5s+L5cygeJAAEFUVCqa+eHhcSZEcInUS5wYDIinYywWMP9xGffYRBYupYkovSTfFwAya1Z3UNpY+geLbYgLQFGywn0/PLqr+cjKpPJHqMCmJGHFv2GmXpzAfN1O7aMbuDWKcYogQfg9AT'
    'kDyuiOLMIb9RTnD+xJKnJS7Ic2IcLTCc19M9gmc2WFaLn5H0hrPpqhyulp9Vz/ssIIVF8+xSZDhZDN1Fjza3eagdsUsBFY3KAuWXVyf+CWgey9WGkxBUQgqo07uLntDc5mlX9rAWbKBNB4BYbJ+4/y'
    'x6nqzo8YETsiW43h4VkvejO/ZJKUaip6B5XKEobCIq+JgpxD9yk+dQIJ1C8oAU/+RNnrVKVnhLEEYUB+pMBfeUv65tKyMlcFEIkF1wkvAkmjxJxSlYEm4j3J0IpJkjY2tkIobHVMjHaPGQ5rnd5trY'
    'Y7vZ59qROvu82206L2L27gPVv2F9tuXSBx32bDgDLw1X69JnYyHzr8JmZOcoWE5WVf98Vl4Cv8k6I23KiLMbVMrBsj+7Xs2v80PXHUpy3jtABCKqpaLh5WyJ3H85m80/QFotFrPFh44LdJ0LGskjXY'
    'LvjQ7c+8tkdTGZ9tl96Fq7nQxy2DE3EEiyhUAtfe1HQML9CJiaBoKLsLJ56wIBQ+woIHQuWHAutUTAdB+v5Ikfsl1id9vdvaY7iRKTJxW1ENxmsXaf6caFz5t5qMM1mTjf0nTaa/pFVY42I27beutu'
    'fY34DpnvsfCikKrMiZCk2tsf8tYLbjRDHRACtbaf9lZabYTWBiDKd+jV4DgAxq1jHkACmdGnZMmcKaLgrtN0HwyaCoAXTRVhYJLumijNKPA+6+9NsAqKRTLHaCi8amq5lmL5FPnWx+75tuYFB7JN3i'
    'ojI2cRVO3VHpYLJX2UiGAmgpZhIa8+z659QGgsoE+xLhI9JSSGtgEhTyMXIpiTgc7Y+eRikC65UArJBRxJ1ugutLXcPyQVfpyFJxBBcKgzzGcda9ze+pSKAMgInkMinrmz+Vv66+aYUX9Z5SQAY1aT'
    'q2qJkK02jh597L24rVIkHqhEFtVy1V+8PwEFQlF98LmjcJTYhhjL1YfPZBY1Jqf71XaQg2pbm3ebt+48JLet4OhBBo7UB3K3xySOKkSOBQfMAaJGFZugdjSYdC5FTE5x9CgUhFooqCVo5sTcfPTo0V'
    'svooWPhEg2Vg1B9GStF9/uiA0IHmV09h2zAJ5/So06KqIlQX43IY6J9LFbdQgnsLOnqHl/W07WmaLwFEFp1Zo6FE8Pa01pU2tqa5fi4/PBccdQtSCGv3BgnwAQdWaD5q4UtetKUe5KOQMjGCWJHPQk'
    'bHAcNIzCbV1IOGHJrYvOZHCSY1nAJmJwKO1gEfXzPjJ4/M24P5nsSVA9qKWYgvPiQuguesIpRE9uhTGnBLdC+YDZ8BNQPRErHZAWfX7ZAlUJdfbzSCc5b02FSN6KN825KAZ+WqpHJe+vOIgeb55t99'
    'eNHKYNHCbcRvWEIuQ3eRjFU24O7KkKP6ue/yXVo0efuj4UUB9B9rTdkTsZIfBxOQ8ewxzB7ZTPgR46fNBACM0HLrXl2QOoxRBSPqaGpGKB4qfXPVGKSATdg/I85fKqu+xppoOobQ+jckiipuTMC/65'
    'd0NuOLsa5EH6y+v5/HKC/9Tag3u258TVXX8nILebUbK5ebVunr1vcdWnzkUI4FJjTF6AYa3VddO12nPfOmDV+RSDd9krdlN87DxhbjNhKiC71TmJ+V1NFEntJoz7nOUg8us97bvXx+qK9H3uPQr32v'
    'GOQ2YIF5BUjlU4RQi9llYkELvmrS0DwRr02fGoxxazjVYk8sCODAvOtS5Kk5M4pN4gkGwhoGIi1UOYb/XGu4OuLaywWDBSBeR1EMtHKNtZYYWBq1HoeRR8Ltzuth+BObUJTbPCo5JKuZhKkur72A2g'
    'S2GWdzs4OSwWxdC0O3cU6LUm/EHfCfB0uKpJUjFX339pQB1mJFiet7KDxhgeD3UsM5ED5nkLObabLqjI1CE2GGHpNIX78v56F2B4UQ1/ukv6WWXIfSXw1irtMXxfaq1rgIPJ1RWWTwywOo4g+/rJpn'
    'v4ADzgzQniPN9pe+VBfMjkfWo3eY9kwyE3xb14F9pOnfMx5gDdm/KbCYFNG4q5HX546Ir4diviYRScHyuRO4Ky1bNvYgotMqnHEAQFh/dyuvVotxyUXxWCG/n8BpN6i9bal7LWzH2HgEoEKTel5gXZ'
    '3Vh90IrEVmaZK9Rt/FGHCHGC0PDeCKpVDzQOH7Qi5FvPvUtAQ1yhIItkuQGk97c4TrAIW5r/oCWcwcwCEOnGZKsoajAr+AIhLkyowZPRKddAWs08RQR0bot4UAWquvYBLYVDcjKTfLJaON3TW3UF8S'
    'flknyWLkFouXUDMah0cD1oLZeP7pnmE2z0SbjEOaYo0E/qYwc2QWFEMZ/zDk4p2mNwCbdzPXH51QpUeVE8oRTi8ATIZKsCPxg2WjCLz0d4oK7UpdbO5GKRfckJDPYWHH0mk/8bMuF2lsTcroZmhNzP'
    'eLbmElQ20TvL+DsQyqMHBUqrfKSAc9HhvIutZ456Xhyzh+IlyQ20+w+QD2fz9bsJL6fnqF/Wh8l7G13QXjkd9UaT5Y+zyXTVq3UBe/NyWl0ui96rWe+yKheofv6SLerNFvmO+WU5rK4q3LW4nhZnz/'
    '549l8k19+r'
)


In [ ]:
# Decode assets and lock the inputs used by the run.
from hashlib import sha256

manifest = 'f58c66ed296444cee8519f2be1e75daad1185fd0655586695486dd7e7fb29525  mod97_local.py\n51b2a57eff34889a975e505b7275d866fc02e9dfef4b374c6fcf1f5927f43a8e  cx/qualify.py\n4828a5c6346d7d1185b067197fd290064f9b9429ed249ba98454ae679ece5f8c  cx/qualify_interfaces.py\n6f0bc98eee2b17ed685e146c98e033138ff18c2b6e6c5df62f6d9829577eed96  cx/qualify_dynamics.py\n91357c62c67bb9cb024d91551adf426cf6d976035de1aae162a25ca17628f4c8  blocks.py\nd9da2449c7b99e45877fabcd54a2c5706ebf3f9bbe7b3f8da184f5ae9a904a49  additional.py\n8a969eca23fd07c534efd3464049aad5c25b37da8f19a8b4d18eb76cdbb60f54  sweep.py\n87c072d49853ed41705f753772aa7be556473b5b864ed2257fe22eb27990c86c  cx/EXTRACT.npz\n4f9b072c10d07780edbe4da223f54d7424ec82cd296550a2bd2b5d622a0edc35  cx/COMPARTMENT_MAP.json\n861342bbf3e243dcd0ea1932b2aabe8687f0f041928952ffc4f44ffe08b037df  cx/RESOLVED_BOUNDARY.json\ne7cb043dd0b1dc43589593d0c489ced3559911a4415adb3788bbfb2c1f21a68a  cx/SYNAPSE_COMPARTMENTS.json\n19d081dbcd53cfad49e28498f211389662abb12929732dca8ed08aa2bc0c1ec1  cx/DYNAMICS_QUALIFICATION.json\n'
for line in manifest.splitlines():
    expected, name = line.split("  ", 1)
    path = WORK / name
    if name in _payload:
        data = zlib.decompress(base64.b64decode(_payload[name], validate=True))
        if sha256(data).hexdigest() != expected:
            raise ValueError(f"Embedded data hash mismatch for {name}")
        path.write_bytes(data)
    if sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f"Source or input hash mismatch for {name}")
(WORK / "SHA256SUMS").write_text(manifest)
del _payload
print("Verified all embedded sources and data.")


## Run and inspect the roster

Each result records the paired test-panel hash, its scope, model outputs or trajectories, and measured CPU time. The retention rows print the still-valid old score after B and also save acquisition and correction. The motion rows print how many probes met their error target and save continuous error metrics. Use the JSON receipts for detailed comparisons.


In [ ]:
runner = [sys.executable, "-B", str(WORK / "sweep.py")]
options = ["--profile", PROFILE, "--out", str(OUT)]
if BLOCKS:
    options += ["--blocks", *BLOCKS]
subprocess.run(runner + ["plan"] + options, cwd=WORK, check=True)
subprocess.run(runner + ["run"] + options + ["--workers", str(WORKERS)], cwd=WORK, check=True)


In [ ]:
subprocess.run(runner + ["report"] + options, cwd=WORK, check=True)
print(f"Per-job JSON receipts are in {OUT / PROFILE}.")
